# GEO Metadata validated and modified process (Gemini-2.0-flash TRUE & Infered has control FALSE)

- ヒトの熱ストレスに関連するRNA-Seqにおいて、Gemini-2.0-flash の判断と自分の判断の結果、解析に使用するに値するデータのメタデータを整理するnotebook
- 整理する際、なぜこのような処理をするのかについても記載する
- `geofetch`から取得したデータと`iseq`から取得したデータ
- [../GenAI_validate/geo_metadata_genai_validation_true_no_control.tsv](../GenAI_validate/geo_metadata_genai_validation_true_no_control.tsv) に分類されたものについてここに記載する

In [1]:
import sys
import polars as pl
import datetime
import pathlib
from typing import Any
sys.path.append(str(pathlib.Path('../../').resolve()))
from scripts.metadata_pairing import create_paired_metadata
from IPython.display import display

pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_cols(100)
pl.Config.set_tbl_rows(4)

polars.config.Config

In [2]:
VERSION = datetime.datetime.now().strftime("%y%m")
execute_date = datetime.datetime.now().strftime("%y%m%d")
print(f"Execute Date: {execute_date}")

# create directory
pathlib.Path(f"../../Data/metadata/metadata_{VERSION}").mkdir(parents=True, exist_ok=True)

Execute Date: 261001


In [3]:
# drop columns list for iseq metadata
drop_columns_list = [
    "sra_aspera",
    "fastq_galaxy",
    "sra_galaxy",
    "fastq_ftp",
    "sra_ftp",
    "fastq_aspera",
    "tax_lineage",
    "bam_md5",
    "fastq_md5",
    "sra_md5"
]

&nbsp;

&nbsp;

&nbsp;

### GSE285529 (SINGLE) NO DOWNLOAD

- validate date 2025-05-02
- これらのデータはヒトのデータをフィルタリングすると熱ストレスを加えていないデータのみ残る
- **今回はこのデータは収集しない**

In [4]:
GSE285529_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE285529/GSE285529_PEP/GSE285529_PEP_raw.csv",
    separator=','
).filter(
    pl.col("sample_organism_ch1") == "Homo sapiens"
)

# display(GSE285529_PEP)

&nbsp;

&nbsp;

&nbsp;

## 1. GSE228346 (SINGLE, ~~PAIRED~~) 

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date 2025-05-02
- validate date 2026-05-04
- [TFIIH kinase CDK7 drives cell proliferation through a common core transcription factor network](https://doi.org/10.1126/sciadv.adr9660)
- 2025年の論文
- nacsent RNAを検出するPRO-seqとRNA-seqのデータが含まれている
- DMSOで処理されたコントロールと42℃で処理された熱ストレスのサンプル (例: `DMSO_0_R1, DMSO_20_R1`) と、CDK阻害剤である `SY5609` を処理された状態でのコントロールと熱ストレスの状態 ( `SY5609_0_R1, SY5609_20_R1`) が存在する
- コントロールと熱ストレスのデータが保存されているので解析に使用する
- PRO-seqとRNA-seqのデータを分けてメタデータを保存しておく
- PRO-seqのデータは[12_7_metadata_proseq.ipynb](./12_7_metadata_proseq.ipynb)に移動
- 複数の温度条件があるので、ペアにした状態のデータフレームに、曝露時間を追記する

&nbsp;

### ストレスの例

- `DMSO_0_R1` → `_0_` 30min pretreatment with DMSO at 37C followed by a 42C heat shock for 0min with a 0min recovery at 37C
- `SY5609_0_R1` → `_0_` 30min pretreatment with SY5609 at 37C followed by a 42C heat shock for 0min with a 0min recovery at 37C
- `	DMSO_30_R1` → `_30_` 30min pretreatment with DMSO at 37C followed by a 42C heat shock for 30min with a 60min recovery at 37C
- `SY5609_30_R1` → `_30_` 30min pretreatment with SY5609 at 37C followed by a 42C heat shock for 30min with a 60min recovery at 37C
- `DMSO_120_R1` → `_120_` 30min pretreatment with DMSO at 37C followed by a 42C heat shock for 120min with a 0min recovery at 37C
- `SY5609_120_R1` → `_120_` 30min pretreatment with SY5609 at 37C followed by a 42C heat shock for 120min with a 0min recovery at 37C

In [5]:
GSE228346_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE228346/GSE228346_PEP/GSE228346_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("KAPA mRNA HyperPrep Kit").alias("rna_library"), # add rna_library column
    pl.lit("https://doi.org/10.1126/sciadv.adr9660").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("42 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("37 C").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# RNA-seq metadata
GSE228346_PEP_rna_seq = GSE228346_PEP.filter(
    (pl.col("sample_library_selection") == "cDNA") &
    (pl.col("sample_library_strategy") == "RNA-Seq")
)

# metadata from iseq
GSE228346_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE228346_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE228346_iseq_filtered = GSE228346_iseq[[s.name for s in GSE228346_iseq if not (s.null_count() == GSE228346_iseq.height)]]

GSE228346_rna_seq_combined = GSE228346_PEP_rna_seq.join(
    GSE228346_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE228346_rna_seq_combined = GSE228346_rna_seq_combined[[s.name for s in GSE228346_rna_seq_combined if not (s.null_count() == GSE228346_rna_seq_combined.height)]]

GSE228346_rna_seq_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE228346_rna_seq_modified_metadata.tsv",
    separator='\t'
)

# display(GSE228346_rna_seq_combined)

In [6]:
# RNA-seq
defined_pairs_gse228346_rna_seq = [
    # DMSO
    ("SRR23989391", "SRR23989385"), # dmso_30_r1 (heat-stress) vs dmso_0_r1 (control)
    ("SRR23989368", "SRR23989386"), # dmso_30_r2 (heat-stress) vs dmso_0_r2 (control)
    ("SRR23989360", "SRR23989387"), # dmso_30_r3 (heat-stress) vs dmso_0_r3 (control)
    ("SRR23989364", "SRR23989385"), # dmso_120_r1 (heat-stress) vs dmso_0_r1 (control)
    ("SRR23989365", "SRR23989386"), # dmso_120_r2 (heat-stress) vs dmso_0_r2 (control)
    ("SRR23989366", "SRR23989387"), # dmso_120_r3 (heat-stress) vs dmso_0_r3 (control)
    # SY5609
    ("SRR23989361", "SRR23989388"), # sy5609_30_r1 (heat-stress) vs sy5609_0_r1 (control)
    ("SRR23989362", "SRR23989389"), # sy5609_30_r2 (heat-stress) vs sy5609_0_r2 (control)
    ("SRR23989363", "SRR23989390"), # sy5609_30_r3 (heat-stress) vs sy5609_0_r3 (control)
    ("SRR23989367", "SRR23989388"), # sy5609_120_r1 (heat-stress) vs sy5609_0_r1 (control)
    ("SRR23989380", "SRR23989389"), # sy5609_120_r2 (heat-stress) vs sy5609_0_r2 (control)
    ("SRR23989379", "SRR23989390")  # sy5609_120_r3 (heat-stress) vs sy5609_0_r3 (control)
]

paired_metadata_df_gse228346_rna_seq = create_paired_metadata(
    combined_df=GSE228346_rna_seq_combined, 
    defined_sample_pairs=defined_pairs_gse228346_rna_seq,
    sample_id_column_name="accession" # default value, but specified to clarify
)



if paired_metadata_df_gse228346_rna_seq is not None:
    paired_metadata_df_gse228346_rna_seq = paired_metadata_df_gse228346_rna_seq.with_columns(
    (pl.when(pl.col("sample_name").str.contains("_30_"))
    .then(pl.lit("30 min."))
    .when(pl.col("sample_name").str.contains("_120_"))
    .then(pl.lit("120 min."))
    .otherwise(pl.lit(""))
    .alias("stress treatment time"))
    )

    pl.Config.set_tbl_rows(2)
    display(paired_metadata_df_gse228346_rna_seq)

    paired_metadata_df_gse228346_rna_seq.write_csv(
        "../../Data/metadata/metadata_paired/GSE228346_rna_seq_paired_metadata.tsv",
        separator='\t'
        )

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_type,treatment,time,assembly,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy_right,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias,stress treatment time
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR23989391""","""SRR23989385""","""dmso_30_r1 vs dmso_0_r1""","""DMSO_30_R1 vs DMSO_0_R1""","""GSM7119261 vs GSM7119255""","""Public on Dec 19 2024""","""Mar 28 2023""","""Dec 19 2024""","""SRA""","""1""","""wild type OV90 cells""","""Homo sapiens""","""9606""","""total RNA""","""RNA-seq from OV90 cells, rnaseq_ercc_normalized_counts.txt""","""GPL24676""","""Taylor,K,Jones""","""tajo5912@colorado.edu""","""3606246612""","""University of Colorado Boulder""","""3415 Colorado Ave""","""Boulder""","""CO""","""80301""","""USA""","""Illumina NovaSeq 6000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM7119261 vs GSM7119255""","""NONE""","""GSE228346""","""0""","""SRX19793297 vs SRX19793283""","""Wild-type OV90 cells""","""ovarian cancer""","""30min pretreatment with DMSO at 37C followed by a 42C heat shock for 30min with a 60min recovery at 37C vs 30min pretreatment with DMSO at 37C followed by a 42C heat shock for 0min with a 0min recover…","""120 vs 30""","""hg38""","""NOT PROVIDED""","""SAMN33946057 vs SAMN33946063""","""KAPA mRNA HyperPrep Kit""","""https://doi.org/10.1126/sciadv.adr9660""","""42 C""","""37 C""","""SRR23989391 vs SRR23989385""","""Illumina NovaSeq 6000 sequencing: GSM7119261: DMSO_30_R1 Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM7119255: DMSO_0_R1 Homo sapiens RNA-Seq""","""The TFIIH kinase CDK7 governs RNA polymerase II initiation, elongation, termination, and RNA processing""","""SRA1611893""","""cDNA""","""2024-12-29""","""9543298184 vs 10127708310""","""6804783092;6902857738 vs 7183867875;7368144859""","""ILLUMINA""","""Generic""","""DMSO_30_R1 vs DMSO_0_R1""","""SRP429713""","""93149347 vs 98216080""","""The TFIIH kinase CDK7 governs RNA polymerase II initiation, elongation, termination, and RNA processing""","""DMSO_30_R1 vs DMSO_0_R1""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: GSM7119261: DMSO_30_R1 Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM7119255: DMSO_0_R1 Homo sapiens RNA-Seq""","""ovarian cancer""","""28131102794 vs 29661256160""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""public""","""GSM7119261 vs GSM7119255""","""GSM7119261_r1 vs GSM7119255_r1""","""RNA-seq: Cells were then harvested by washing wit

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE272629 (SINGLE?) No Download

- validate date 2025-05-03
- [Passive shaping of intra- and intercellular m6A dynamics via mRNA metabolism](https://doi.org/10.7554/eLife.100448.2)
- `Camptothecin (CPT)` 処理を含むデータ、`Actinomycin D (ActD)`処理を含むデータが登録されている
- ヒト培養細胞のデータを見ると熱ストレス曝露のデータは含まれていないので今回は解析に使用しない

In [7]:
# GSE272629_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE272629/GSE272629_PEP/GSE272629_PEP_raw.csv",
#     separator=','
# ).filter(
#     pl.col("sample_organism_ch1") == "Homo sapiens"
# )

# # display(GSE272629_PEP)

### GSE103233 (SINGLE?) No Download

- validate date 2025-05-03
- RNA-seqのデータはこれは記述を見ても熱ストレス曝露のデータではなかったので今回は収集しない

In [8]:
# GSE103233_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE103233/GSE103233_PEP/GSE103233_PEP_raw.csv",
#     separator=','
# ).filter(
#     pl.col("sample_library_strategy") == "RNA-Seq"
# )

# # display(GSE103233_PEP)

### GSE116472 (SINGLE,PAIRED) NO DOWNLOAD

- validate date 2025-05-03
- [Single-molecule nascent RNA sequencing identifies regulatory domain architecture at promoters and enhancers](https://doi.org/10.1038/s41588-018-0234-5)
- ヒトだけのデータをフィルタリングしたところ、熱ストレス曝露のデータではなかったので、今回は収集しない

In [9]:
# GSE116472_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE116472/GSE116472_PEP/GSE116472_PEP_raw.csv",
#     separator=','
# ).filter(
#     pl.col("sample_organism_ch1") == "Homo sapiens"
# )

# # display(GSE116472_PEP)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2. GSE255207 (SINGLE) 

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date 2025-05-03
- [Dynamics of RNA localization to nuclear speckles are connected to splicing efficiency](https://doi.org/10.1126/sciadv.adp7727)
- `Nuclear speckles`という細胞小器官にあるRNAに関する研究
- 熱ストレスに曝露されたデータに加え、スプライシング阻害剤である`Pladienolide B (pladB)`処理を加えているサンプルが存在するのでこれをフィルタリングする
- また、`HepG2`細胞のサンプルは熱ストレスのデータが存在していないので、これもフィルタリングする
- `For heat shock treatment, HeLa cells were stressed at 43°C for 2 h.`
- MockとHeat shockのデータを収集する

In [10]:
GSE255207_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE255207/GSE255207_PEP/GSE255207_PEP_raw.csv",
    separator=','
).filter(
    ~(pl.col("treatment") == "Pladienolide B treated") & # splicing inhibitor
    ~(pl.col("cell_line") == "HepG2")
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("SMARTer Stranded Total RNA-Seq Kit v2").alias("rna_library"), # add rna_library column (Takara)
    pl.lit("https://doi.org/10.1126/sciadv.adp7727").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("43 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("120 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE255207_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE255207_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE255207_iseq_filtered = GSE255207_iseq[[s.name for s in GSE255207_iseq if not (s.null_count() == GSE255207_iseq.height)]]

GSE255207_combined = GSE255207_PEP.join(
    GSE255207_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE255207_combined = GSE255207_combined[[s.name for s in GSE255207_combined if not (s.null_count() == GSE255207_combined.height)]]

GSE255207_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE255207_modified_metadata.tsv",
    separator='\t'
)

# display(GSE255207_combined)

In [11]:
defined_pairs_gse255207 = [
    ("SRR27892100", "SRR27892104"), # hela_heatshock_rep1_polya_rna_seq (heat-stress) vs hela_mock_rep1_polya_rna_seq (control)
    ("SRR27892099", "SRR27892103"), # hela_heatshock_rep2_polya_rna_seq (heat-stress) vs hela_mock_rep2_polya_rna_seq (control)
    ("SRR27892092", "SRR27892096"), # hela_heatshock_rep1_nuclear_rna_seq (heat-stress) vs hela_mock_rep1_nuclear_rna_seq (control)
    ("SRR27892091", "SRR27892095"), # hela_heatshock_rep2_nuclear_rna_seq (heat-stress) vs hela_mock_rep2_nuclear_rna_seq (control)
]

paired_metadata_df_gse255207 = create_paired_metadata(
    combined_df=GSE255207_combined,
    defined_sample_pairs=defined_pairs_gse255207,
    sample_id_column_name="accession"
)


if paired_metadata_df_gse255207 is not None:
    paired_metadata_df_gse255207.write_csv(
        "../../Data/metadata/metadata_paired/GSE255207_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse255207)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_laboratory,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_type,genotype,treatment,assembly,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,country,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR27892100""","""SRR27892104""","""hela_heatshock_rep1_polya_rna_seq vs hela_mock_rep1_polya_rna_seq""","""HeLa_HeatShock_rep1_polyA_RNA_seq vs HeLa_mock_rep1_polyA_RNA_seq""","""GSM8065769 vs GSM8065765""","""Public on Sep 05 2024""","""Feb 06 2024""","""Sep 05 2024""","""SRA""","""1""","""Cervical cancer cells""","""Homo sapiens""","""9606""","""polyA RNA""","""GPL24676""","""Jingyi,,Fei""","""jingyifei@uchicago.edu""","""7737024370""","""Fei Lab""","""The University of Chicago""","""GCIS W142, 929 E 57th Street""","""Chicago""","""IL""","""60637""","""USA""","""Illumina NovaSeq 6000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM8065769 vs GSM8065765""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8065nnn/GSM8065769/suppl/GSM8065769_HeLa_HeatShock_rep1_polyA_RNA_seq.count.txt.gz vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8065nnn/GSM8065765/suppl/GSM80…","""GSE255207, GSE255208""","""0""","""SRX23552598 vs SRX23552594""","""HeLa""","""Cervical cancer cells""","""WT""","""Heat shock vs Mock""","""hg38""","""SAMN39845943 vs SAMN39845947""","""SMARTer Stranded Total RNA-Seq Kit v2""","""https://doi.org/10.1126/sciadv.adp7727""","""43 C""","""120 min.""","""37 C (inferred)""","""SRR27892100 vs SRR27892104""","""Illumina NovaSeq 6000 sequencing: GSM8065769: HeLa_HeatShock_rep1_polyA_RNA_seq Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM8065765: HeLa_mock_rep1_polyA_RNA_seq Homo sapiens RNA-Seq""","""Dynamics of RNA localization to nuclear speckles are connected to splicing efficiency [RNA-Seq]""","""SRA1798768""","""cDNA""","""2024-09-29""","""1632612867 vs 1640637555""","""2416890547 vs 2462126115""","""ILLUMINA""","""missing""","""Generic""","""HeLa_HeatShock_rep1_polyA_RNA_seq vs HeLa_mock_rep1_polyA_RNA_seq""","""SRP488515""","""42691730 vs 43616606""","""Dynamics of RNA localization to nuclear speckles are connected to splicing efficiency [RNA-Seq]""","""HeLa_HeatShock_rep1_polyA_RNA_seq vs HeLa_mock_rep1_polyA_RNA_seq""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: GSM8065769: HeLa_HeatShock_rep1_polyA_RNA_seq Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM8065765: HeLa_mock_rep1_polyA_RNA_seq Homo sapiens RNA-Seq""","""Cervical cancer cells""","""51656

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 3. GSE270672 (SINGLE) 

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date 2025-05-03
- [An HSF1–JMJD6–HSP feedback circuit promotes cell adaptation to proteotoxic stress](https://doi.org/10.1073/pnas.2313370121)
- HSF1の活性化に関与する因子としてJMJD6を同定した論文で、silencingされたデータや、プロテアソーム阻害剤 MG132のデータが存在する
- `siJMJD6`の熱ショックを加えたデータは、コントロールがなかったので今回は収集しない
- [Appendix 01 (PDF)](https://www.pnas.org/doi/suppl/10.1073/pnas.2313370121/suppl_file/pnas.2313370121.sapp.pdf) にストレス条件が記載されている
- `siNT`の`basal` と`Heatshock42C(1h)`のデータを収集する

In [12]:
GSE270672_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE270672/GSE270672_PEP/GSE270672_PEP_raw.csv",
    separator=','
).filter(
    ((pl.col("treatment") == "Basal") | (pl.col("treatment") == "Heat shock 42C (1h), 2h Recovery")) &
    ~(pl.col("genotype") == "siJMJD6")
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("Lexogen QuantSeq FWD Kit").alias("rna_library"), # add rna_library column (Lexogen) from supplemetary information (S1 appendix)
    pl.lit("https://doi.org/10.1073/pnas.2313370121").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("42 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("60 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE270672_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE270672_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE270672_iseq_filtered = GSE270672_iseq[[s.name for s in GSE270672_iseq if not (s.null_count() == GSE270672_iseq.height)]]

GSE270672_combined = GSE270672_PEP.join(
    GSE270672_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE270672_combined = GSE270672_combined[[s.name for s in GSE270672_combined if not (s.null_count() == GSE270672_combined.height)]]

GSE270672_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE270672_modified_metadata.tsv",
    separator='\t'
)

# display(GSE270672_combined)

In [13]:
defined_pairs_gse270672 = [
    ("SRR29525025", "SRR29525028"), # ma_pc3m_sint_hs_1 (Heat shock 42C (1h), 2h Recovery) vs ma_pc3m_sint_1 (Basal)
    ("SRR29525024", "SRR29525027"), # ma_pc3m_sint_hs_2 (Heat shock 42C (1h), 2h Recovery) vs ma_pc3m_sint_2 (Basal)
    ("SRR29525023", "SRR29525026")  # ma_pc3m_sint_hs_3 (Heat shock 42C (1h), 2h Recovery) vs ma_pc3m_sint_3 (Basal)
]

paired_metadata_df_gse270672 = create_paired_metadata(
    combined_df=GSE270672_combined,
    defined_sample_pairs=defined_pairs_gse270672,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse270672 is not None:
    paired_metadata_df_gse270672.write_csv(
        "../../Data/metadata/metadata_paired/GSE270672_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse270672)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_type,genotype,treatment,assembly,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,country,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR29525025""","""SRR29525028""","""ma_pc3m_sint_hs_1 vs ma_pc3m_sint_1""","""MA_PC3M_siNT_HS_1 vs MA_PC3M_siNT_1""","""GSM8349806 vs GSM8349803""","""Public on Jul 18 2024""","""Jun 24 2024""","""Jul 18 2024""","""SRA""","""1""","""PC3M""","""Homo sapiens""","""9606""","""Cells were grown at 37°C and 5% CO2""","""total RNA""","""GPL18573""","""Marc,L,Mendillo""","""mendillo@northwestern.edu""","""3125034862""","""Marc Mendillo""","""Biochemistry and Molecular Genetics""","""Northwestern University""","""Chicago""","""IL""","""60611""","""USA""","""Illumina NextSeq 500""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM8349806 vs GSM8349803""","""NONE""","""GSE270672""","""0""","""SRX25035093 vs SRX25035090""","""PC3M""","""Prostate cancer cells""","""siNT""","""Heat shock 42C (1h), 2h Recovery vs Basal""","""hg38""","""SAMN42017738 vs SAMN42017741""","""Lexogen QuantSeq FWD Kit""","""https://doi.org/10.1073/pnas.2313370121""","""42 C""","""60 min.""","""37 C (inferred)""","""SRR29525025 vs SRR29525028""","""NextSeq 500 sequencing: GSM8349806: MA_PC3M_siNT_HS_1 Homo sapiens RNA-Seq vs NextSeq 500 sequencing: GSM8349803: MA_PC3M_siNT_1 Homo sapiens RNA-Seq""","""An HSF1-JMJD6-HSP feedback circuit promotes cell adaptation to proteotoxic stress [RNA-Seq]""","""SRA1906783""","""cDNA""","""2024-07-23""","""89646818 vs 61627464""","""124380977 vs 86322699""","""ILLUMINA""","""missing""","""Generic""","""MA_PC3M_siNT_HS_1 vs MA_PC3M_siNT_1""","""SRP515692""","""2683691 vs 1821251""","""An HSF1-JMJD6-HSP feedback circuit promotes cell adaptation to proteotoxic stress [RNA-Seq]""","""MA_PC3M_siNT_HS_1 vs MA_PC3M_siNT_1""","""NextSeq 500""","""NextSeq 500 sequencing: GSM8349806: MA_PC3M_siNT_HS_1 Homo sapiens RNA-Seq vs NextSeq 500 sequencing: GSM8349803: MA_PC3M_siNT_1 Homo sapiens RNA-Seq""","""Prostate cancer cells""","""201907067 vs 137162696""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""GSM8349806 vs GSM8349803""","""GSM8349806_r1 vs GSM8349803_r1""","""RNA was extracted using the Qiagen RNeasy kit according to supplied protocol. Libraries were prepared from 100 ng RNA using the Lexogen QuantSeq FWD kit for Illumina sequencing , according to supplied…","""missin

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE255206 (SINGLE) NO DOWNLOAD

- validate date:2025-05-03
- [Dynamics of RNA localization to nuclear speckles are connected to splicing efficiency](https://doi.org/10.1126/sciadv.adp7727)
- ARTR-seq (reverse transcription–based RNA binding protein binding sites sequencing) という手法のデータ
- ARTR-Seqの論文: [Profiling of RNA-binding protein binding sites by in situ reverse transcription-based sequencing](https://doi.org/10.1038/s41592-023-02146-w)
- GSE255207の論文の別のプロジェクト
- 今回はこのデータは収集しない

### GSE173537 (SINGLE) PRO-seq

- validate date: 2025-05-03
- [Characterizing primary transcriptional responses to short term heat shock in Down syndrome](https://doi.org/10.1371/journal.pone.0307375)
- 全てPRO-seqのデータで、chromosome 21 trisomic のデータも登録されている
- [12_7_metadata_proseq.ipynb](./12_7_metadata_proseq.ipynb)に処理を移動

### GSE197992 (SINGLE) TT-chem-seq

- validation date: 2025-05-03
- [An Important Role for RPRD1B in the Heat Shock Response](https://pmc.ncbi.nlm.nih.gov/articles/PMC9583720/)
- おそらく上記の論文由来のデータ
- RPRD1Bの機能を調査しているデータ
- TTchem-seqのデータのプロジェクト
- コントロールデータと熱ストレス曝露のデータが揃っている
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に処理を移動

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 4. GSE197993 (SINGLE)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-05-03
- validation date: 2026-05-04
- [An Important Role for RPRD1B in the Heat Shock Response](https://pmc.ncbi.nlm.nih.gov/articles/PMC9583720/)
- おそらく上記の論文由来のデータ (このデータはGEOのsubseries)
- RPRD1Bの機能を調査しているデータ
- こちらはRNA-seqのデータのプロジェクト
- コントロールデータと熱ストレス曝露のデータが揃っている
- 曝露時間についてはメタデータに記載があったものの、温度条件が見つからない
- [Supplementary Material](https://pmc.ncbi.nlm.nih.gov/articles/instance/9583720/bin/mcb.00173-22-s0008.pdf) にキットや温度条件の記載がある
- 収集する

In [ ]:
GSE197993_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE197993/GSE197993_PEP/GSE197993_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("KAPA mRNA HyperPrep Kit").alias("rna_library"), # add rna_library column (Illumina)
    pl.lit("https://doi.org/10.1128/mcb.00173-22").alias("reference_doi"), # add DOI of the reference paper column (間違っていたので修正)
    pl.lit("43 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("120 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE197993_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE197993_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE197993_iseq_filtered = GSE197993_iseq[[s.name for s in GSE197993_iseq if not (s.null_count() == GSE197993_iseq.height)]]

GSE197993_combined = GSE197993_PEP.join(
    GSE197993_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE197993_combined = GSE197993_combined[[s.name for s in GSE197993_combined if not (s.null_count() == GSE197993_combined.height)]]

GSE197993_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE197993_modified_metadata.tsv",
    separator='\t'
)

# display(GSE197993_combined)

In [15]:
defined_pairs_gse197993 = [
    ("SRR18248924", "SRR18248925"), # HS_siNT_B1 [RNA-seq] (120min heat shock) vs NHS_siNT_B1 [RNA-seq] (No heat shock)
    ("SRR18248920", "SRR18248921"), # HS_siNT_B2 [RNA-seq] (120min heat shock) vs NHS_siNT_B2 [RNA-seq] (No heat shock)
    ("SRR18248916", "SRR18248917"), # HS_siNT_B3 [RNA-seq] (120min heat shock) vs NHS_siNT_B3 [RNA-seq] (No heat shock)
    ("SRR18248922", "SRR18248923"), # HS_siRPRD1B_B1 [RNA-seq] (120min heat shock) vs NHS_siRPRD1B_B1 [RNA-seq] (No heat shock)
    ("SRR18248918", "SRR18248919"), # HS_siRPRD1B_B2 [RNA-seq] (120min heat shock) vs NHS_siRPRD1B_B2 [RNA-seq] (No heat shock)
    ("SRR18248914", "SRR18248915")  # HS_siRPRD1B_B3 [RNA-seq] (120min heat shock) vs NHS_siRPRD1B_B3 [RNA-seq] (No heat shock)
]

paired_metadata_df_gse197993 = create_paired_metadata(
    combined_df=GSE197993_combined,
    defined_sample_pairs=defined_pairs_gse197993,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse197993 is not None:
    paired_metadata_df_gse197993.write_csv(
        "../../Data/metadata/metadata_paired/GSE197993_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse197993)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,sirna,condition,genome_build,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR18248924""","""SRR18248925""","""hs_sint_b1_rna-seq_ vs nhs_sint_b1_rna-seq_""","""HS_siNT_B1 [RNA-seq] vs NHS_siNT_B1 [RNA-seq]""","""GSM5935415 vs GSM5935414""","""Public on Sep 08 2022""","""Mar 06 2022""","""Sep 08 2022""","""SRA""","""1""","""MRC5-VA cell line""","""Homo sapiens""","""9606""","""polyA RNA""","""CUG4409A2, mRNA-Seq, poly(A)+ mRNA vs CUG4409A1, mRNA-Seq, poly(A)+ mRNA""","""GPL24676""","""Richard,James,Mitter""","""Bioinformatics & Biostatistics""","""The Francis Crick Institute""","""1 Midland Road""","""London""","""LONDON""","""NW1 1AT""","""United Kingdom""","""Illumina NovaSeq 6000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM5935415 vs GSM5935414""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5935nnn/GSM5935415/suppl/GSM5935415_HS_siNT_B1.bigWig vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5935nnn/GSM5935414/suppl/GSM5935414_NHS_siNT_B1.bigWig""","""GSE197993, GSE197995""","""0""","""SRX14390290 vs SRX14390289""","""MRC5-VA""","""non-targeting siRNA""","""120min heat shock vs No heat shock""","""GRCh38""","""SAMN26491009 vs SAMN26491010""","""KAPA mRNA HyperPrep Kit""","""https://doi.org/10.1016/j.molcel.2022.01.007""","""43 C""","""120 min.""","""37 C (inferred)""","""SRR18248924 vs SRR18248925""","""Illumina NovaSeq 6000 sequencing: GSM5935415: HS_siNT_B1 [RNA-seq] Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM5935414: NHS_siNT_B1 [RNA-seq] Homo sapiens RNA-Seq""","""RPRD1B is required for proper heat shock response [RNA-seq]""","""SRA1381503""","""cDNA""","""2022-09-09""","""1553090531 vs 1868564775""","""2407037431 vs 2986226469""","""ILLUMINA""","""HS_siNT_B1 [RNA-seq] vs NHS_siNT_B1 [RNA-seq]""","""SRP362755""","""50011912 vs 60393279""","""RPRD1B is required for proper heat shock response [RNA-seq]""","""HS_siNT_B1 [RNA-seq] vs NHS_siNT_B1 [RNA-seq]""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: GSM5935415: HS_siNT_B1 [RNA-seq] Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM5935414: NHS_siNT_B1 [RNA-seq] Homo sapiens RNA-Seq""","""5051203112 vs 6099721179""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""GSM5935415 vs GSM5935414""","""GSM5935415_r1 vs GSM5935414_r1""","""RNA was extracted using the RNeasy kit (Qiagen, 74104) according to the manufacturer's instruction including the on-column DNase treatment. 200ng 

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE189562 (SINGLE) TT-chem-seq

- validation date 2025-05-03
- [Heat shock induces premature transcript termination and reconfigures the human transcriptome](https://doi.org/10.1016/j.molcel.2022.01.007)
- 以前の研究でも使用していたTTchem-seqのデータ
- 一つのGSM IDに対して2つのSRAのrun IDが付与されている (`srr`カラムが`multi`)
- 念の為収集する
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に処理を移動

### GSE165365 (SINGLE) (DREB) TT-chem-seq

- validation date: 2025-05-03
- [Heat shock induces premature transcript termination and reconfigures the human transcriptome](https://doi.org/10.1016/j.molcel.2022.01.007)
- このデータも以前の研究で使用したDRB_TTchem-seqのデータ
- コントロールと熱ストレス曝露のデータが揃っているので念の為収集
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に処理を移動

### GSE127844 (SINGLE) PRO-seq

- validate date: 2025-05-03
- [Stress-induced transcriptional memory accelerates promoter-proximal pause release and decelerates termination over mitotic divisions](https://doi.org/10.1016/j.molcel.2021.03.007)
- 以前の研究で使用したPRO-seqのデータ
- HSF1のsilencing 処理も含まれている
- コントロールデータが含まれているので念の為収集
- 処理を[12_7_metadata_proseq.ipynb](./12_7_metadata_proseq.ipynb)に移動

### GSE130493 (SINGLE) NO DOWNLOAD

- validate date: 2025-05-03
- [rDNA Clusters Make Contact with Genes that Are Involved in Differentiation and Cancer and Change Contacts after Heat Shock Treatment](https://doi.org/10.3390/cells8111393)
- 熱ストレスに曝露されているデータは登録されているが、コントロールデータがないため、今回は使用しない

In [16]:
# GSE130493_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE130493/GSE130493_PEP/GSE130493_PEP_raw.csv",
#     separator=','
# )

# display(GSE130493_PEP)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 5. GSE137559 (SINGLE)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-05-03
- [17β-Estradiol Activates HSF1 via MAPK Signaling in ERα-Positive Breast Cancer Cells](https://doi.org/10.3390/cancers11101533)
- これも以前の研究で使用した研究
- estrogen処理のデータのみフィルタリング
- `Materials and Methods` の `4.3 Treatments` に記載がある

In [17]:
GSE137559_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE137559/GSE137559_PEP/GSE137559_PEP_raw.csv",
    separator=','
).filter(
    ~pl.col("sample_title").str.contains(r'estrogen') # remove estrogen data
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("unknown").alias("rna_library"), # add rna_library column
    pl.lit("https://doi.org/10.3390/cancers11101533").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("43 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("60 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature"), # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE137559_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE137559_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE137559_iseq_filtered = GSE137559_iseq[[s.name for s in GSE137559_iseq if not (s.null_count() == GSE137559_iseq.height)]]

GSE137559_combined = GSE137559_PEP.join(
    GSE137559_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE137559_combined = GSE137559_combined[[s.name for s in GSE137559_combined if not (s.null_count() == GSE137559_combined.height)]]

GSE137559_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE137559_modified_metadata.tsv",
    separator='\t'
)

# display(GSE137559_combined)

In [18]:
defined_pairs_gse137559 = [
    ("SRR10129832", "SRR10129830") # RNA-seq_hyperthermia_2h vs RNA-seq_control
]

paired_metadata_df_gse137559 = create_paired_metadata(
    combined_df=GSE137559_combined,
    defined_sample_pairs=defined_pairs_gse137559,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse137559 is not None:
    paired_metadata_df_gse137559.write_csv(
        "../../Data/metadata/metadata_paired/GSE137559_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse137559)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,tumor,sex,age,tissue_cells,genome_build,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR10129832""","""SRR10129830""","""rna-seq_hyperthermia_2h vs rna-seq_control""","""cDNA""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR10129832 vs SRR10129830""","""SRX6858245 vs SRX6858243""","""RNA-seq_hyperthermia_2h vs RNA-seq_control""","""GSM4081765 vs GSM4081763""","""Public on Oct 05 2019""","""Sep 17 2019""","""Oct 05 2019""","""SRA""","""1""","""cells subjected to hyperhermia (43°C) for 1hour & 2hours of recovery vs untreated control cells""","""Homo sapiens""","""9606""","""total RNA""","""cells subjected to hyperhermia (43°C) for 1hour & 2hours of recovery, NG.7926_wt_hs_lib80493_4027_1_1 vs untreated control cells, NG.7926_wt_k_lib80492_4027_8_1""","""GPL16791""","""Tomasz,,Stokowy""","""tomasz.stokowy@uib.no""","""Scientific Computing""","""IT Division""","""University of Bergen""","""Nygardsgaten 5""","""Bergen""","""5020""","""Norway""","""Illumina HiSeq 2500""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM4081765 vs GSM4081763""","""NONE""","""GSE137559, GSE137560""","""0""","""SRX6858245 vs SRX6858243""","""MCF7""","""adenocarcinoma""","""female""","""69 years""","""breast""","""hg19""","""SAMN12771812 vs SAMN12771814""","""unknown""","""https://doi.org/10.3390/cancers11101533""","""43 C""","""60 min.""","""37 C (inferred)""","""SRR10129832 vs SRR10129830""","""Illumina HiSeq 2500 sequencing: GSM4081765: RNA-seq_hyperthermia_2h Homo sapiens RNA-Seq vs Illumina HiSeq 2500 sequencing: GSM4081763: RNA-seq_control Homo sapiens RNA-Seq""","""Heat Shock Factor 1 (HSF1) acquires transcriptional competence under 17b-estradiol in ERa-positive breast cancer cells [RNA-seq]""","""SRA963111""","""cDNA""","""2019-10-07""","""1399135241 vs 888674258""","""1913941906 vs 1191500893""","""ILLUMINA""","""Generic""","""RNA-seq_hyperthermia_2h vs RNA-seq_control""","""SRP221987""","""74855306 vs 46705889""","""Heat Shock Factor 1 (HSF1) acquires transcriptional competence under 17b-estradiol in ERa-positive breast cancer cells [RNA-seq]""","""RNA-seq_hyperthermia_2h vs RNA-seq_control""","""Illumina HiSeq 2500""","""Illumina HiSeq 2500 sequencing: GSM4081765: RNA-seq_hyperthermia_2h Homo sapiens RNA-Seq vs Illumina HiSeq 2500 sequencing: GSM4081763: RNA-seq_control Homo 

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 6. GSE130892 RNA-seq (SINGLE)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION (RNA-seq)

&nbsp;

- validation date: 2025-05-03
- [Limits in the detection of m6A changes using MeRIP/m6A-seq](https://doi.org/10.1038/s41598-020-63355-3)
- 以前の研究で使用していたRNA-seqとMeRIP-seq (methylated RNA immunoprecipitation sequencing) のデータ
- MeRIP-seq の処理を[./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に移動
- memo: 2026-06-26
- salmon での 発現定量の際に、以下のようなエラーメッセージを確認したため、`SRR9029566` と `SRR9029568` をスキップする
- `  # error message: [2026-01-17 21:36:25.085] [jointLog] [error] Processing reads : Error reading from the FASTA/Q stream. Minimum return code for left and right read was (-2). Make sure the file is valid.`
- GitHub issue: https://github.com/COMBINE-lab/salmon/issues/660

In [19]:
GSE130892_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE130892/GSE130892_PEP/GSE130892_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("unknown").alias("rna_library"), # add rna_library column (Illumina) ?
    pl.lit("https://doi.org/10.1038/s41598-020-63355-3").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("43 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("60 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# RNA-seq
GSE130892_PEP_rna_seq = GSE130892_PEP.filter(
    pl.col("library_strategy") == "RNA-seq"
)

# metadata from iseq
GSE130892_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE130892_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE130892_iseq_filtered = GSE130892_iseq[[s.name for s in GSE130892_iseq if not (s.null_count() == GSE130892_iseq.height)]]

GSE130892_combined_rna_seq = GSE130892_PEP_rna_seq.join(
    GSE130892_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True)

GSE130892_combined_rna_seq.write_csv(
    "../../Data/metadata/metadata_modified/GSE130892_rna_seq_modified_metadata.tsv",
    separator='\t'
)

# display(GSE130892_combined_rna_seq)

In [20]:
# RNA-seq
defined_pairs_gse130892_rna_seq = [
    # ("SRR9029562", "SRR9029568"), # heatshock_total_input_1 vs untreated_total_input_1 # due to error, skip
    ("SRR9029564", "SRR9029570"), # heatshock_total_input_2 vs untreated_total_input_2
    # ("SRR9029566", "SRR9029572")  # heatshock_total_input_3 vs untreated_total_input_3 # due to error, skip
]

paired_metadata_df_gse130892_rna_seq = create_paired_metadata(
    combined_df=GSE130892_combined_rna_seq,
    defined_sample_pairs=defined_pairs_gse130892_rna_seq,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse130892_rna_seq is not None:
    paired_metadata_df_gse130892_rna_seq.write_csv(
        "../../Data/metadata/metadata_paired/GSE130892_rna_seq_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse130892_rna_seq)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_type,treatment,library_strategy,antibody,genome_build,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy_right,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR9029564""","""SRR9029570""","""heatshock_total_input_2 vs untreated_total_input_2""","""cDNA""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR9029564 vs SRR9029570""","""SRX5806989 vs SRX5806995""","""heatshock_total_input_2 vs untreated_total_input_2""","""GSM3755850 vs GSM3755856""","""Public on May 09 2019""","""May 08 2019""","""May 09 2019""","""SRA""","""1""","""OCI-Ly1 diffuse B-cell lymphoma cells""","""Homo sapiens""","""9606""","""total RNA""","""TRIzol, Illumina, 1 x 42""","""heatshock_total_2.macs2_peaks.narrowPeak vs untreated_total_2.macs2_peaks.narrowPeak""","""GPL16791""","""A,,McIntyre""","""Molecular Life Sciences""","""University of Zurich""","""Winterthurerstrasse 190""","""Zurich""","""8057""","""Switzerland""","""Illumina HiSeq 2500""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM3755850 vs GSM3755856""","""NONE""","""GSE130892""","""0""","""SRX5806989 vs SRX5806995""","""OCI-Ly1""","""diffuse B-cell lymphoma cells""","""43 C for 1 hr, recovery at 37 C for 1 hr vs 37 C""","""RNA-seq""","""none""","""hg38""","""MACS2 peak files (bed)""","""SAMN11604635 vs SAMN11604629""","""unknown""","""https://doi.org/10.1038/s41598-020-63355-3""","""43 C""","""60 min.""","""37 C (inferred)""","""SRR9029564 vs SRR9029570""","""Illumina HiSeq 2500 sequencing: GSM3755850: heatshock_total_input_2 Homo sapiens RNA-Seq vs Illumina HiSeq 2500 sequencing: GSM3755856: untreated_total_input_2 Homo sapiens RNA-Seq""","""MeRIP-seq for heat shock in B-cell lymphoma cells""","""SRA884323""","""cDNA""","""2019-05-10""","""1104929167 vs 1011890323""","""1604039274 vs 1467495955""","""ILLUMINA""","""Generic""","""heatshock_total_input_2 vs untreated_total_input_2""","""SRP197091""","""36851164 vs 33344708""","""MeRIP-seq for heat shock in B-cell lymphoma cells""","""heatshock_total_input_2 vs untreated_total_input_2""","""Illumina HiSeq 2500""","""Illumina HiSeq 2500 sequencing: GSM3755850: heatshock_total_input_2 Homo sapiens RNA-Seq vs Illumina HiSeq 2500 sequencing: GSM3755856: untreated_total_input_2 Homo sapiens RNA-Seq""","""diffuse B-cell lymphoma cells""","""1547748888 vs 1400477736""","""RNA-Seq""","""TRANS

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE69021 (SINGLE) NO DOWNLOAD

- validation date: 2025-05-03
- [Gene target specificity of the Super Elongation Complex (SEC) family: how HIV-1 Tat employs selected SEC members to activate viral transcription](https://doi.org/10.1093/nar/gkv541)
- AFF4、AFF1に対してRNAiを行っているデータ
- 熱ストレスを加えているのはChIP-seqで、このRNA-seqのデータはKDをしているので収集しない

In [21]:
# GSE69021_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE69021/GSE69021_PEP/GSE69021_PEP_raw.csv",
#     separator=','
# )

# display(GSE69021_PEP)

### GSE57397 (SINGLE) NO DOWNLOAD

- validation date: 2025-05-03
- [FBXW7 modulates cellular stress response and metastatic potential through HSF1 post-translational modification](https://doi.org/10.1038/ncb3121)
- 熱ストレス時のデータと熱ストレス後の回復のデータのみ登録されているので今回は使用しない

In [22]:
# GSE57397_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE57397/GSE57397_PEP/GSE57397_PEP_raw.csv",
#     separator=','
# )

# display(GSE57397_PEP)

###  GSE37002 (SINGLE) NO DOWNLOAD

- validation date: 2025-05-03
- [Topology of the human and mouse m6A RNA methylomes revealed by m6A-seq](https://doi.org/10.1038/nature11112)
- m6A-seqも以前使用したが、ChIP-seqのような使用方法のため、今回は収集しない

In [23]:
# GSE37002_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE37002/GSE37002_PEP/GSE37002_PEP_raw.csv",
#     separator=','
# )

# display(GSE37002_PEP)

### GSE211651 (SINGLE) PRO-seq

- validation date: 2025-05-03
- [Enhancing HIV-1 latency reversal through regulating the elongating RNA Pol II pause-release by a small-molecule disruptor of PAF1C](https://doi.org/10.1126/sciadv.adf2468)
- きちんと熱ストレス条件とコントロール条件があるデータ
- `inhibitor of PAF1C (iPAF1C)` や `shRNA` 処理も重ねて行われているデータも
- **RNA-seqのデータを見ると、熱ストレスに曝露されているデータは含まれていなかったので、PRO-seqのみ念の為収集しておく**
- 処理を[12_7_metadata_proseq.ipynb](./12_7_metadata_proseq.ipynb)に移動

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 7. GSE132447 (PAIRED) RNA-seq

- ✅ DOWNLOAD 
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-05-03
- [Multiple Alu Exonization in 3′UTR of a Primate-Specific Isoform of CYP20A1 Creates a Potential miRNA Sponge](https://doi.org/10.1093/gbe/evaa233)
-  HIV1-Tat treatment (おそらくtat)のデータをフィルタリングして、RNA-seqとsmall RNA-seqのデータを収集する
- `HS was given at 45°C (±0.2)  for 30 mins in a water bath.`
- small RNA-seqのデータの処理は[./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に移動

In [24]:
GSE132447_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE132447/GSE132447_PEP/GSE132447_PEP_raw.csv",
    separator=','
)

GSE132447_PEP_rna_seq = GSE132447_PEP.filter(
    pl.col("sample_library_strategy") == "RNA-Seq" # filter out small RNA-seq data
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("TruSeq stranded total RNA protocol").alias("rna_library"), # add rna_library column from GEO webpage (Illumina)
    pl.lit("https://doi.org/10.1093/gbe/evaa233").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("45 C (±0.2)").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("30 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE132447_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE132447_metadata.tsv",
    separator='\t'
).with_columns(
    pl.col("cell_line").fill_null(pl.col("cell_type")) # use cell_type (Primary neurons) when cell_line is empty
).drop(
    drop_columns_list
)

GSE132447_iseq_filtered = GSE132447_iseq[[s.name for s in GSE132447_iseq if not (s.null_count() == GSE132447_iseq.height)]]

GSE132447_combined_rna_seq = GSE132447_PEP_rna_seq.join(
    GSE132447_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
).filter(
    ~(pl.col("sample_title").str.contains(r'_Tat_')) # filter out HIV1-Tat treatment data
)

GSE132447_combined_rna_seq = GSE132447_combined_rna_seq[[s.name for s in GSE132447_combined_rna_seq if not (s.null_count() == GSE132447_combined_rna_seq.height)]]

GSE132447_combined_rna_seq.write_csv(
    "../../Data/metadata/metadata_modified/GSE132447_rna_seq_modified_metadata.tsv",
    separator='\t'
)

# display(GSE132447_combined_rna_seq)

In [25]:
# RNA-seq
defined_pairs_gse132447_rna_seq = [
    ("SRR9257619", "SRR9257616"), # RNA seq_Heat_shock_1 hr recovery_Replicate_1 vs RNA seq_Untreated_Replicate_1
    ("SRR9257620", "SRR9257617"), # RNA seq_Heat_shock_1 hr recovery_Replicate_2 vs RNA seq_Untreated_Replicate_2
    ("SRR9257621", "SRR9257618"), # RNA seq_Heat_shock_1 hr recovery_Replicate_3 vs RNA seq_Untreated_Replicate_3
    ("SRR9257622", "SRR9257616"), # RNA seq_Heat_shock_3 hr recovery_Replicate_1 vs RNA seq_Untreated_Replicate_1
    ("SRR9257623", "SRR9257617"), # RNA seq_Heat_shock_3 hr recovery_Replicate_2 vs RNA seq_Untreated_Replicate_2
    ("SRR9257624", "SRR9257618"), # RNA seq_Heat_shock_3 hr recovery_Replicate_3 vs RNA seq_Untreated_Replicate_3
    ("SRR9257625", "SRR9257616"), # RNA seq_Heat_shock_24 hr recovery_Replicate_1 vs RNA seq_Untreated_Replicate_1
    ("SRR9257626", "SRR9257617"), # RNA seq_Heat_shock_24 hr recovery_Replicate_2 vs RNA seq_Untreated_Replicate_2
    ("SRR9257627", "SRR9257618")  # RNA seq_Heat_shock_24 hr recovery_Replicate_3 vs RNA seq_Untreated_Replicate_3
]

paired_metadata_df_gse132447_rna_seq = create_paired_metadata(
    combined_df=GSE132447_combined_rna_seq,
    defined_sample_pairs=defined_pairs_gse132447_rna_seq,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse132447_rna_seq is not None:
    paired_metadata_df_gse132447_rna_seq.write_csv(
        "../../Data/metadata/metadata_paired/GSE132447_rna_seq_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse132447_rna_seq)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_type,genome_build,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR9257619""","""SRR9257616""","""rna_seq_heat_shock_1_hr_recovery_replicate_1 vs rna_seq_untreated_replicate_1""","""cDNA""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR9257619 vs SRR9257616""","""SRX6028080 vs SRX6028077""","""RNA seq_Heat_shock_1 hr recovery_Replicate_1 vs RNA seq_Untreated_Replicate_1""","""GSM3864915 vs GSM3864912""","""Public on Jan 21 2021""","""Jun 10 2019""","""Jan 21 2021""","""SRA""","""1""","""Primary neurons""","""Homo sapiens""","""9606""","""total RNA""","""GPL11154""","""vineet,,jha""","""Persistent system""","""LABS""","""Persistent systems""","""9 A/12 Pingale Aryabhatta Persistent Tower,""","""pune""","""Maharashtra""","""411004""","""India""","""Illumina HiSeq 2000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM3864915 vs GSM3864912""","""NONE""","""GSE132447""","""0""","""SRX6028080 vs SRX6028077""","""Primary neurons""","""hg19""","""cuffdiff output""","""SAMN12001297 vs SAMN12001389""","""TruSeq stranded total RNA protocol""","""https://doi.org/10.1093/gbe/evaa233""","""45 C (±0.2)""","""30 min.""","""37 C (inferred)""","""SRR9257619 vs SRR9257616""","""Illumina HiSeq 2000 sequencing: GSM3864915: RNA seq_Heat_shock_1 hr recovery_Replicate_1 Homo sapiens RNA-Seq vs Illumina HiSeq 2000 sequencing: GSM3864912: RNA seq_Untreated_Replicate_1 Homo sapiens …","""Origin of a novel CYP20A1 lncRNA through 23 Alu exaptations in the human lineage creates a potential multi-miRNA sponge""","""SRA896538""","""cDNA""","""2021-01-23""","""11503475023 vs 11982557292""","""6756195292;6831003726 vs 6891036998;6978775702""","""ILLUMINA""","""Generic""","""RNA seq_Heat_shock_1 hr recovery_Replicate_1 vs RNA seq_Untreated_Replicate_1""","""SRP200969""","""80197951 vs 84947575""","""Origin of a novel CYP20A1 lncRNA through 23 Alu exaptations in the human lineage creates a potential multi-miRNA sponge""","""RNA seq_Heat_shock_1 hr recovery_Replicate_1 vs RNA seq_Untreated_Replicate_1""","""Illumina HiSeq 2000""","""Illumina HiSeq 2000 sequencing: GSM3864915: RNA seq_Heat_shock_1 hr recovery_Replicate_1 Homo sapiens RNA-Seq vs Illumina HiSeq 2000 sequencing: GSM3864912: RNA seq_Untreated_Replicate_1 Homo sapiens …","""Primary neurons""","""14802635789 vs 15670357006""","

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE294487 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-03
- 論文はアクセスできないが、貴重なので念の為収集しておこう... と思ったが、アセンブリーがmm10など情報がおかしいものや、hyperthermiaの条件がはっきりわからなかったので今回は収集しない

In [26]:
# GSE294487_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE294487/GSE294487_PEP/GSE294487_PEP_raw.csv",
#     separator=','
# )

# display(GSE294487_PEP)

### GSE261649 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-03
- [Pre-print: Multi-omics and biochemical reconstitution reveal CDK7-dependent mechanisms controlling RNA polymerase II function at gene 5’- and 3’-ends](https://doi.org/10.1101/2025.01.08.632016)
- 全てのデータが熱ショックが加えられており、今回は解析には使用しない

In [27]:
# GSE261649_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE261649/GSE261649_PEP/GSE261649_PEP_raw.csv",
#     separator=','
# )

# display(GSE261649_PEP)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 8. GSE289231 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-05-03
- [Differential Temperature-Induced Responses in Immortalized Oral and Skin Keratinocytes](https://doi.org/10.3390/ijms26072851)
- コントロールとなる37℃, 熱ストレスとなる60℃, 低温ストレスとなる-25℃でそれぞれ10分間に曝露されたデータ
- 低温ストレスのデータをフィルタリングした状態で収集

In [28]:
GSE289231_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE289231/GSE289231_PEP/GSE289231_PEP_raw.csv",
    separator=','
).filter(
    ~pl.col("sample_title").str.contains(r'(-25)')
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("unknown").alias("rna_library"), # add rna_library (performed by Novogene)
    pl.lit("https://doi.org/10.3390/ijms26072851").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("60 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("10 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE289231_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE289231_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE289231_iseq_filtered = GSE289231_iseq[[s.name for s in GSE289231_iseq if not (s.null_count() == GSE289231_iseq.height)]]

GSE289231_combined = GSE289231_PEP.join(
    GSE289231_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE289231_combined = GSE289231_combined[[s.name for s in GSE289231_combined if not (s.null_count() == GSE289231_combined.height)]]

GSE289231_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE289231_modified_metadata.tsv",
    separator='\t'
)

# display(GSE289231_combined)

In [29]:
defined_pairs_gse289231 = [
    # Skin Keratinocytes
    ("SRR32325145", "SRR32325151"), # HaCaT 60C-1 vs HaCaT 37C-1
    ("SRR32325144", "SRR32325150"), # HaCaT 60C-2 vs HaCaT 37C-2
    ("SRR32325143", "SRR32325149"), # HaCaT 60C-3 vs HaCaT 37C-3
    # Gingival Keratinocytes
    ("SRR32325136", "SRR32325142"), # TIGK 60C-1 vs TIGK 37C-1
    ("SRR32325135", "SRR32325141"), # TIGK 60C-2 vs TIGK 37C-2
    ("SRR32325134", "SRR32325140")  # TIGK 60C-3 vs TIGK 37C-3
]

paired_metadata_df_gse289231 = create_paired_metadata(
    combined_df=GSE289231_combined,
    defined_sample_pairs=defined_pairs_gse289231,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse289231 is not None:
    paired_metadata_df_gse289231.write_csv(
        "../../Data/metadata/metadata_paired/GSE289231_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse289231)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,tissue,cell_line,cell_type,treatment,assembly,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,country,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias,tissue_type
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR32325145""","""SRR32325151""","""hacat_60c-1 vs hacat_37c-1""","""HaCaT 60C-1 vs HaCaT 37C-1""","""GSM8787406 vs GSM8787400""","""Public on Apr 17 2025""","""Feb 10 2025""","""Apr 17 2025""","""SRA""","""1""","""Skin""","""Homo sapiens""","""9606""","""polyA RNA""","""Library name: Sample 7 vs Library name: Sample 1""","""GPL24676""","""Chen,,Han""","""chan206@uic.edu""","""3018288051""","""Luisa DiPietro""","""Center for Wound healing and Tissue Regeneration""","""University of Illinois Chicago""","""801 S. Paulina St.""","""Chicago""","""IL""","""60612""","""USA""","""Illumina NovaSeq 6000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM8787406 vs GSM8787400""","""NONE""","""GSE289231""","""0""","""SRX27660171 vs SRX27660165""","""Skin""","""HaCaT""","""Skin Keratinocytes""","""60°C Heat Injury vs 37°C Baseline""","""hg38""","""SAMN46758295 vs SAMN46758301""","""unknown""","""https://doi.org/10.3390/ijms26072851""","""60 C""","""10 min.""","""37 C""","""SRR32325145 vs SRR32325151""","""Illumina NovaSeq 6000 sequencing: GSM8787406: HaCaT 60C-1 Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM8787400: HaCaT 37C-1 Homo sapiens RNA-Seq""","""DIFFERENTIAL TEMPERATURE-INDUCED RESPONSES IN ORAL AND SKIN KERATINOCYTES""","""SRA2076099""","""cDNA""","""2025-04-18""","""2945660261 vs 3969268696""","""2158940969;2158669115 vs 2934485513;2872989603""","""ILLUMINA""","""null""","""null""","""SRP563552""","""34032118 vs 44657457""","""DIFFERENTIAL TEMPERATURE-INDUCED RESPONSES IN ORAL AND SKIN KERATINOCYTES""","""null""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: GSM8787406: HaCaT 60C-1 Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM8787400: HaCaT 37C-1 Homo sapiens RNA-Seq""","""null""","""10209635400 vs 13397237100""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""public""","""null""","""GSM8787406_r1 vs GSM8787400_r1""","""RNA was harvested after 10 minutes in -25°C, 37°C, and 60°C using Trizol and purified with an RNA Clean & Concentrator-25 kit (Zymo, Tustin, CA, USA), and treated with DNAse (ThermoFisher Scientific, …","""null""","""GSM8787406_r1 vs GSM8787400_r1""","""null""","""GSM8787406 vs GSM8787400""","""2025-

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 9. GSE279569 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-05-03
- [Histone lactylation drives liver cancer metastasis by facilitating NSF1-mediated ferroptosis resistance after microwave ablation](https://doi.org/10.1016/j.redox.2025.103553)
- コントロールとなる37℃,15分のデータと47℃,15分のデータが揃っているので、このデータを収集して、他のデータはフィルタリングする

In [30]:
GSE279569_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE279569/GSE279569_PEP/GSE279569_PEP_raw.csv",
    separator=','
).filter(
    ~((pl.col("sample_title").str.contains(r'NC-')) | (pl.col("sample_title").str.contains(r'NFS1'))) # filter out NC- and NFS1-treated samples
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("VAHTS Universal V6 RNA-seq Library Prep Kit").alias("rna_library"), # add rna_library column from GEO webpage (VAHTS)
    pl.lit("https://doi.org/10.1016/j.redox.2025.103553").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("47 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("15 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE279569_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE279569_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE279569_iseq_filtered = GSE279569_iseq[[s.name for s in GSE279569_iseq if not (s.null_count() == GSE279569_iseq.height)]]

GSE279569_combined = GSE279569_PEP.join(
    GSE279569_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE279569_combined = GSE279569_combined[[s.name for s in GSE279569_combined if not (s.null_count() == GSE279569_combined.height)]]

GSE279569_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE279569_modified_metadata.tsv",
    separator='\t'
)

# display(GSE279569_combined)

In [31]:
defined_pairs_gse279569 = [
    ("SRR31013091", "SRR31013096"), # 47C-15min-1 vs 37C-15min-1
    ("SRR31013090", "SRR31013095"), # 47C-15min-2 vs 37C-15min-2
    ("SRR31013089", "SRR31013094"), # 47C-15min-3 vs 37C-15min-3
    ("SRR31013088", "SRR31013093"), # 47C-15min-4 vs 37C-15min-4
    ("SRR31013087", "SRR31013092")  # 47C-15min-5 vs 37C-15min-5
]

paired_metadata_df_gse279569 = create_paired_metadata(
    combined_df=GSE279569_combined,
    defined_sample_pairs=defined_pairs_gse279569,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse279569 is not None:
    paired_metadata_df_gse279569.write_csv(
        "../../Data/metadata/metadata_paired/GSE279569_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse279569)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,tissue,cell_line,cell_type,genotype,treatment,assembly,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,country,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias,tissue_type
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR31013091""","""SRR31013096""","""r-47-1 vs r-37-1""","""R-47-1 vs R-37-1""","""GSM8576137 vs GSM8576132""","""Public on Mar 12 2025""","""Oct 16 2024""","""Mar 12 2025""","""SRA""","""1""","""liver cancer cell""","""Homo sapiens""","""9606""","""polyA RNA""","""GPL24676""","""Jia,yan,Huang""","""hjy8103@scu.edu.cn""","""Ultrasound""","""West China hospital""","""No.37 Guoxue Lane, Wuhou District""","""Sichuan""","""610041""","""China""","""Illumina NovaSeq 6000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM8576137 vs GSM8576132""","""NONE""","""GSE279569""","""0""","""SRX26400560 vs SRX26400555""","""liver cancer cell""","""MHCC-97H""","""human hepatocellular carcinoma cells""","""WT""","""The samples were exposed to a water bath at 47℃ for 15 minutes and subsequently incubated in a standard cell incubator for 24 hours. vs The samples were exposed to a water bath at 37℃ for 15 minutes a…","""Hg38""","""SAMN44308030 vs SAMN44308035""","""VAHTS Universal V6 RNA-seq Library Prep Kit""","""https://doi.org/10.1016/j.redox.2025.103553""","""47 C""","""15 min.""","""37 C""","""SRR31013091 vs SRR31013096""","""Illumina NovaSeq 6000 sequencing: GSM8576137: R-47-1 Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM8576132: R-37-1 Homo sapiens RNA-Seq""","""Histone lactylation drives liver cancer metastasis by facilitating NSF1 mediated ferroptosis resistance after microwave ablation""","""SRA1992484""","""cDNA""","""2025-03-14""","""1748428614 vs 1717410082""","""1303049896;1301704108 vs 1276026205;1275138572""","""ILLUMINA""","""missing""","""R-47-1 vs R-37-1""","""SRP538866""","""22967753 vs 22655054""","""Histone lactylation drives liver cancer metastasis by facilitating NSF1 mediated ferroptosis resistance after microwave ablation""","""R-47-1 vs R-37-1""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: GSM8576137: R-47-1 Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM8576132: R-37-1 Homo sapiens RNA-Seq""","""human hepatocellular carcinoma cells""","""6856598699 vs 6763408540""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""public""","""GSM8576137 vs GSM8576132""","""GSM8576137_r1 vs GSM8576132_r1""","""Total RNA was extracted using the TRIzol reagent (Invitrogen, CA, USA) according to the manufacturer's protocol

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 10. GSE285497 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-05-03
- [Heat shock-induced PI(4)P increase drives HSPA1A translocation to the plasma membrane in cancer and stressed cells through PI4KIII alpha activation](https://doi.org/10.1016/j.cstres.2025.100130)
- 42℃ 60分間と37℃のデータが存在する
- `treatment` が `heat shock`で `0hr`と `_8hr`があるが、reocover が8時間している場合のデータもある
- `Cells were either maintained at 37°C or subjected to heat stress at 42°C for 60 minutes in a humidified CO₂ incubator.`
- 解析に使用する

In [32]:
GSE285497_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE285497/GSE285497_PEP/GSE285497_PEP_raw.csv",
    separator=','
).filter(
    ~((pl.col("sample_title").str.contains(r'NC-')) | (pl.col("sample_title").str.contains(r'NFS1'))) # filter out NC- and NFS1-treated samples
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("NEBNext® UltraTM RNA Library Prep Kit for Illumina® (NEB, USA)").alias("rna_library"), # add rna_library column from GEO webpage
    pl.lit("https://doi.org/10.1016/j.cstres.2025.100130").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("42 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("60 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE285497_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE285497_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE285497_iseq_filtered = GSE285497_iseq[[s.name for s in GSE285497_iseq if not (s.null_count() == GSE285497_iseq.height)]]

GSE285497_combined = GSE285497_PEP.join(
    GSE285497_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE285497_combined = GSE285497_combined[[s.name for s in GSE285497_combined if not (s.null_count() == GSE285497_combined.height)]]

GSE285497_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE285497_modified_metadata.tsv",
    separator='\t'
)

# display(GSE285497_combined)

In [33]:
defined_pairs_gse285497 = [
    # human embryonic kidney cells
    ("SRR31847460", "SRR31847461"), # hek_1_1_0hr vs hek_1_1_control
    ("SRR31847459", "SRR31847461"), # hek_1_1_8hr vs hek_1_1_control
    ("SRR31847457", "SRR31847458"), # hek_1_1_b2_0hr vs hek_1_1_b2_control
    ("SRR31847456", "SRR31847458"), # hek_1_1_b2_8hr vs hek_1_1_b2_control
    ("SRR31847454", "SRR31847455"), # hek_2_1_0hr vs hek_2_1_control
    ("SRR31847453", "SRR31847455"), # hek_2_1_8hr vs hek_2_1_control
    ("SRR31847451", "SRR31847452"), # hek_2_1_b2_0hr vs hek_2_1_b2_control
    ("SRR31847450", "SRR31847452"), # hek_2_1_b2_8hr vs hek_2_1_b2_control
    ("SRR31847448", "SRR31847449"), # hek_3_1_0hr vs hek_3_1_control
    ("SRR31847447", "SRR31847449"), # hek_3_1_8hr vs hek_3_1_control
    ("SRR31847445", "SRR31847446"), # hek_3_1_b2_0hr vs hek_3_1_b2_control
    ("SRR31847444", "SRR31847446"), # hek_3_1_b2_8hr vs hek_3_1_b2_control
    ("SRR31847442", "SRR31847443"), # hek_4_1_0hr vs hek_4_1_control
    ("SRR31847441", "SRR31847443"), # hek_4_1_8hr vs hek_4_1_control
    ("SRR31847439", "SRR31847440"), # hek_5_1_0hr vs hek_5_1_control
    ("SRR31847438", "SRR31847440"), # hek_5_1_8hr vs hek_5_1_control
    ("SRR31847436", "SRR31847437"), # hek_6_1_0hr vs hek_6_1_control
    ("SRR31847435", "SRR31847437"), # hek_6_1_8hr vs hek_6_1_control
    # Henrietta Lacks cells
    ("SRR31847433", "SRR31847434"), # hela_1_1_0hr vs hela_1_1_control
    ("SRR31847432", "SRR31847434"), # hela_1_1_8hr vs hela_1_1_control
    ("SRR31847430", "SRR31847431"), # hela_1_1_b2_0hr vs hela_1_1_b2_control
    ("SRR31847429", "SRR31847431"), # hela_1_1_b2_8hr vs hela_1_1_b2_control
    ("SRR31847427", "SRR31847428"), # hela_2_1_0hr vs hela_2_1_control
    ("SRR31847426", "SRR31847428"), # hela_2_1_8hr vs hela_2_1_control
    ("SRR31847424", "SRR31847425"), # hela_2_1_b2_0hr vs hela_2_1_b2_control
    ("SRR31847422", "SRR31847423"), # hela_3_1_0hr vs hela_3_1_control
    ("SRR31847421", "SRR31847423"), # hela_3_1_8hr vs hela_3_1_control
    ("SRR31847419", "SRR31847420"), # hela_3_1_b2_8hr vs hela_3_1_b2_control
    ("SRR31847417", "SRR31847418"), # hela_4_1_0hr vs hela_4_1_control
    ("SRR31847416", "SRR31847418"), # hela_4_1_8hr vs hela_4_1_control
    ("SRR31847414", "SRR31847415"), # hela_5_1_0hr vs hela_5_1_control
    ("SRR31847413", "SRR31847415"), # hela_5_1_8hr vs hela_5_1_control
    ("SRR31847411", "SRR31847412"), # hela_6_1_0hr vs hela_6_1_control
    ("SRR31847410", "SRR31847412"), # hela_6_1_8hr vs hela_6_1_control
    # hepatocellular carcinoma cells
    ("SRR31847408", "SRR31847409"), # hep_1_1_0hr vs hep_1_1_control
    ("SRR31847407", "SRR31847409"), # hep_1_1_8hr vs hep_1_1_control
    ("SRR31847405", "SRR31847406"), # hep_1_1_b2_0hr vs hep_1_1_b2_control
    ("SRR31847404", "SRR31847406"), # hep_1_1_b2_8hr vs hep_1_1_b2_control
    ("SRR31847402", "SRR31847403"), # hep_2_1_0hr vs hep_2_1_control
    ("SRR31847401", "SRR31847403"), # hep_2_1_8hr vs hep_2_1_control
    ("SRR31847399", "SRR31847400"), # hep_2_1_b2_0hr vs hep_2_1_b2_control
    ("SRR31847398", "SRR31847400"), # hep_2_1_b2_8hr vs hep_2_1_b2_control
    ("SRR31847396", "SRR31847397"), # hep_3_1_0hr vs hep_3_1_control
    ("SRR31847395", "SRR31847397"), # hep_3_1_8hr vs hep_3_1_control
    ("SRR31847393", "SRR31847394"), # hep_3_1_b2_0hr vs hep_3_1_b2_control
    ("SRR31847392", "SRR31847394"), # hep_3_1_b2_8hr vs hep_3_1_b2_control
    ("SRR31847390", "SRR31847391"), # hep_4_1_0hr vs hep_4_1_control
    ("SRR31847389", "SRR31847391"), # hep_4_1_8hr vs hep_4_1_control
    ("SRR31847387", "SRR31847388"), # hep_5_1_0hr vs hep_5_1_control
    ("SRR31847386", "SRR31847388"), # hep_5_1_8hr vs hep_5_1_control
    ("SRR31847384", "SRR31847385"), # hep_6_1_0hr vs hep_6_1_control
    ("SRR31847383", "SRR31847385"), # hep_6_1_8hr vs hep_6_1_control
]

paired_metadata_df_gse285497 = create_paired_metadata(
    combined_df=GSE285497_combined,
    defined_sample_pairs=defined_pairs_gse285497,
    sample_id_column_name="accession"
)


if paired_metadata_df_gse285497 is not None:
    paired_metadata_df_gse285497.write_csv(
        "../../Data/metadata/metadata_paired/GSE285497_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse285497)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,tissue,cell_line,cell_type,genotype,treatment,batch,library_name,column,assembly,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,country,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias,tissue_type
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR31847460""","""SRR31847461""","""hek_1_1_0hr vs hek_1_1_control""","""HEK_1_1;0hr vs HEK_1_1;control""","""GSM8703440 vs GSM8703439""","""Public on Dec 31 2024""","""Dec 29 2024""","""Dec 31 2024""","""SRA""","""1""","""human embryonic kidney cells""","""Homo sapiens""","""9606""","""total RNA""","""GPL24676""","""Nikolas,,Nikolaidis""","""nnikolaidis@fullerton.edu""","""Biological Science""","""California State Univeristy Fullerton""","""800 N. State College Blvd""","""Fullerton""","""CA""","""92831""","""USA""","""Illumina NovaSeq 6000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM8703440 vs GSM8703439""","""NONE""","""GSE285497""","""0""","""SRX27207323 vs SRX27207322""","""human embryonic kidney cells""","""HEK293 cells""","""human embryonic kidney cells""","""wild type""","""heatshock vs no heatshock""","""1""","""HEK1_0R_1_B1 vs HEK1_cnt_1_B1""","""HEK7_0R vs HEK7cnt""","""hg38""","""SAMN46011606 vs SAMN46011607""","""NEBNext® UltraTM RNA Library Prep Kit for Illumina® (NEB, USA)""","""https://doi.org/10.1016/j.cstres.2025.100130""","""42 C""","""60 min.""","""37 C""","""SRR31847460 vs SRR31847461""","""Illumina NovaSeq 6000 sequencing: GSM8703440: HEK_1_1 0hr Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM8703439: HEK_1_1 control Homo sapiens RNA-Seq""","""Transcriptomics Unveil Canonical and Non-Canonical Heat Shock-Induced Pathways in Human Cell Lines""","""SRA2042349""","""cDNA""","""2025-01-29""","""3885795543 vs 3786605688""","""2905399196;2962522271 vs 2831721321;2903082280""","""ILLUMINA""",null,null,"""SRP554280""","""40629763 vs 39926163""","""Transcriptomics Unveil Canonical and Non-Canonical Heat Shock-Induced Pathways in Human Cell Lines""",null,"""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: GSM8703440: HEK_1_1 0hr Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM8703439: HEK_1_1 control Homo sapiens RNA-Seq""",null,"""12188928900 vs 11977848900""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""public""",null,"""GSM8703440_r1 vs GSM8703439_r1""","""Cells from the control, 0R, and 8R conditions were sent to Novogene (Sacramento, CA) for RNA isolation, library preparation, and sequencing. Total RNA was isolated using the 

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE247043 (PAIRED) single-cell RNA-seq

- validation date: 2025-05-03
- [Pre-print: Spatiotemporal Single-Cell Rna Sequencing Reveals that Cd4+T Subpopulation with High Metabolic Flux Plays Pivotal Roles Under in Hyperthermia Treatment Warts](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=4636648)
- これはsingle-cell RNA-seq (scRNA-seq) のデータ
- [./12_2_metadata_modified_true_no_control.ipynb](./12_2_metadata_modified_true_no_control.ipynb)に処理を移動

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 11. GSE279445 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-05-03
- [Ultrasound Control of Genomic Regulatory Toolboxes for Cancer Immunotherapy](https://doi.org/10.1038/s41467-024-54477-7)
- Nalm6 cells という細胞種に対してbulk RNA-seqを行っているデータ
- 43℃ 10 min. で高温に曝露しているデータと、コントロールのデータが存在するので解析に使用する

In [34]:
GSE279445_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE279445/GSE279445_PEP/GSE279445_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("unknown").alias("rna_library"), # add rna_library column
    pl.lit("https://doi.org/10.1038/s41467-024-54477-7").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("43 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("10 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE279445_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE279445_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE279445_iseq_filtered = GSE279445_iseq[[s.name for s in GSE279445_iseq if not (s.null_count() == GSE279445_iseq.height)]]

GSE279445_combined = GSE279445_PEP.join(
    GSE279445_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE279445_combined = GSE279445_combined[[s.name for s in GSE279445_combined if not (s.null_count() == GSE279445_combined.height)]]

GSE279445_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE279445_modified_metadata.tsv",
    separator='\t'
)

# display(GSE279445_combined)

In [35]:
defined_pairs_gse279445 = [
    ("SRR30989156", "SRR30989157"), # N6215HS24h vs N6215CT24h
    ("SRR30989154", "SRR30989155"), # N6203HS24h vs N6203CT24h
    ("SRR30989152", "SRR30989153"), # N6215HS48h vs N6215CT48h
    ("SRR30989150", "SRR30989151"), # N6203HS48h vs N6203CT48h
    ("SRR30989148", "SRR30989149"), # N6215HS96h vs N6215CT96h
    ("SRR30989146", "SRR30989147")  # N6203HS96h vs N6203CT96h
]

paired_metadata_df_gse279445 = create_paired_metadata(
    combined_df=GSE279445_combined,
    defined_sample_pairs=defined_pairs_gse279445,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse279445 is not None:
    paired_metadata_df_gse279445.write_csv(
        "../../Data/metadata/metadata_paired/GSE279445_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse279445)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_characteristics_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_institute,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,assembly,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,country,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR30989156""","""SRR30989157""","""n6215hs24h vs n6215ct24h""","""N6215HS24h vs N6215CT24h""","""GSM8570762 vs GSM8570761""","""Public on Oct 14 2024""","""Oct 14 2024""","""Oct 14 2024""","""SRA""","""1""","""Nalm6""","""Homo sapiens""","""9606""","""cell line: Nalm6""","""total RNA""","""GPL16791""","""Kun,,Sun""","""sunkun@szbl.ac.cn""","""+8675586967710""","""Shenzhen Bay Laboratory""","""Shenzhen""","""Guangdong""","""518132""","""China""","""Illumina HiSeq 2500""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM8570762 vs GSM8570761""","""NONE""","""GSE279445""","""0""","""SRX26377450 vs SRX26377449""","""NCBI GRCh38""","""Raw read counts for all genes""","""SAMN44280146 vs SAMN44280147""","""unknown""","""https://doi.org/10.1038/s41467-024-54477-7""","""43 C""","""10 min.""","""37 C""","""SRR30989156 vs SRR30989157""","""Illumina HiSeq 2500 sequencing: GSM8570762: N6215HS24h Homo sapiens RNA-Seq vs Illumina HiSeq 2500 sequencing: GSM8570761: N6215CT24h Homo sapiens RNA-Seq""","""Effect of HS-mediated telomere disruption on Nalm6 cells""","""SRA1991569""","""cDNA""","""2024-10-16""","""2541093158 vs 2463570088""","""1825243179;1919994789 vs 1784176388;1856428557""","""ILLUMINA""","""missing""","""Generic""","""N6215HS24h vs N6215CT24h""","""SRP538492""","""23732879 vs 23243914""","""Effect of HS-mediated telomere disruption on Nalm6 cells""","""N6215HS24h vs N6215CT24h""","""Illumina HiSeq 2500""","""Illumina HiSeq 2500 sequencing: GSM8570762: N6215HS24h Homo sapiens RNA-Seq vs Illumina HiSeq 2500 sequencing: GSM8570761: N6215CT24h Homo sapiens RNA-Seq""","""7119863700 vs 6973174200""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""public""","""GSM8570762 vs GSM8570761""","""GSM8570762_r1 vs GSM8570761_r1""","""Total RNA was collected from the engineered cells at 24, 48, and 96 h after HS using the RNA microprep kit (Zymo Research, R1050) and sent for bulk RNA-seq (Novogene) Library was constructed by Novoge…","""missing""","""GSM8570762_r1 vs GSM8570761_r1""","""Nalm6""","""GSM8570762 vs GSM8570761""","""2024-10-16""","""PRJNA1172720""","""SRS22901288 vs SRS22901287""","""2024-10-16""","""GSE279445"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""SRR30989

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE234710 (PAIRED) (BRB-seq)

- validate date: 2025-05-03
- validate date: 2026-05-04
- Bulk RNA barcoding sequencing (BRB) という手法だが、メタデータを見ると、`RNA-Seq` となっている
- [Transcriptional landscape of human keratinocyte models exposed to 60-GHz millimeter-waves](https://doi.org/10.1016/j.tiv.2024.105808)
- `Sham unexposed cells(Sham)`, `millimeter waves (MMW) exposed Cells (Expo)`, `heat shock control (HSC)`の3つの実験条件 + `HEK, NHEK, HaCaT` の3つの細胞種の条件
- 温度条件は論文の [Table 1](https://www.sciencedirect.com/science/article/pii/S0887233324000389?via%3Dihub#t0005), Table 3 に記載されている
- `sham` は 36℃前後であることから、これをコントロールとして使用できると考えられる
- ~~熱を加えている、という側面があるため、全てのデータを使用する~~
- **`heat shock control (HSC)` と `sham` のみ 収集する方針に転換**
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に移動

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 12. GSE229915 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date: 2025-05-03
- [Heat promotes melanogenesis by increasing the paracrine effects in keratinocytes via the TRPV3/Ca2+/Hh signaling pathway](https://doi.org/10.1016/j.isci.2023.106749)
- 熱ストレスとメラニン生成の関係を調査している論文
- 37℃, 41℃のデータがそれぞれしっかり揃っているので解析に使用する

In [36]:
GSE229915_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE229915/GSE229915_PEP/GSE229915_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("TruSeq Stranded mRNA LT Sample Prep Kit").alias("rna_library"), # add rna_library column from GEO webpage (Illumina)
    pl.lit("https://doi.org/10.1016/j.isci.2023.106749").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("41 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("72 h.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C").alias("control treatment temperature"), # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE229915_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE229915_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE229915_iseq_filtered = GSE229915_iseq[[s.name for s in GSE229915_iseq if not (s.null_count() == GSE229915_iseq.height)]]

GSE229915_combined = GSE229915_PEP.join(
    GSE229915_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE229915_combined = GSE229915_combined[[s.name for s in GSE229915_combined if not (s.null_count() == GSE229915_combined.height)]]

GSE229915_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE229915_modified_metadata.tsv",
    separator='\t'
)

# display(GSE229915_combined)

In [37]:
defined_pairs_gse229915 = [
    ("SRR24196212", "SRR24196215"), #heat_41_1 vs heat_37_1
    ("SRR24196211", "SRR24196214"), #heat_41_2 vs heat_37_2
    ("SRR24196210", "SRR24196213")  #heat_41_3 vs heat_37_3
]

paired_metadata_df_gse229915 = create_paired_metadata(
    combined_df=GSE229915_combined,
    defined_sample_pairs=defined_pairs_gse229915,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse229915 is not None:
    paired_metadata_df_gse229915.write_csv(
        "../../Data/metadata/metadata_paired/GSE229915_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse229915)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_institute,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,tissue,cell_line,cell_type,genotype,treatment,assembly,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,country,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias,tissue_type
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR24196212""","""SRR24196215""","""heat_41_1 vs heat_37_1""","""heat_41_1 vs heat_37_1""","""GSM7180386 vs GSM7180383""","""Public on May 20 2023""","""Apr 17 2023""","""May 20 2023""","""SRA""","""1""","""skin""","""Homo sapiens""","""9606""","""total RNA""","""GPL16791""","""Lan,,Zhang""","""208311035@csu.edu.cn""","""18384660986""","""中南大学""","""Changsha""","""--- Select One ---""","""410006""","""China""","""Illumina HiSeq 2500""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM7180386 vs GSM7180383""","""NONE""","""GSE229915""","""0""","""SRX19992957 vs SRX19992954""","""skin""","""HaCaT""","""human immortalized keratinocyte""","""not applicable""","""41℃ for 3 days, 1 hour per day vs 37℃ for 3 days, 1 hour per day""","""NCBI_GRCh38.p13""","""SAMN34224446 vs SAMN34224449""","""TruSeq Stranded mRNA LT Sample Prep Kit""","""https://doi.org/10.1016/j.isci.2023.106749""","""41 C""","""72 h.""","""37 C""","""SRR24196212 vs SRR24196215""","""Illumina HiSeq 2500 sequencing: GSM7180386: heat_41_1 Homo sapiens RNA-Seq vs Illumina HiSeq 2500 sequencing: GSM7180383: heat_37_1 Homo sapiens RNA-Seq""","""Heat promotes melanogenesis by increasing the paracrine effects in keratinocytes via the TRPV3/Ca2+/Hh signaling pathway""","""SRA1622553""","""cDNA""","""2023-06-05 vs 2023-06-16""","""1849290853 vs 1766625496""","""1396129887;1396479674 vs 1334487895;1334742792""","""ILLUMINA""","""missing""","""Generic""","""heat_41_1 vs heat_37_1""","""SRP433036""","""25297280 vs 24137300""","""Heat promotes melanogenesis by increasing the paracrine effects in keratinocytes via the TRPV3/Ca2+/Hh signaling pathway""","""heat_41_1 vs heat_37_1""","""Illumina HiSeq 2500""","""Illumina HiSeq 2500 sequencing: GSM7180386: heat_41_1 Homo sapiens RNA-Seq vs Illumina HiSeq 2500 sequencing: GSM7180383: heat_37_1 Homo sapiens RNA-Seq""","""human immortalized keratinocyte""","""7044759111 vs 6732394610""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""public""","""GSM7180386 vs GSM7180383""","""GSM7180386_r1 vs GSM7180383_r1""","""Total RNA was extracted using the mirVana miRNA Isolation Kit (Ambion) following the manufacturer's protocol. The libraries were constructed using TruSeq Stranded mRNA LTSample Prep Kit (Illumina, San…","""missing""","""GSM7180386_r1 vs GSM7

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 13. GSE242710 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date: 2025-05-03
- [Synergetic support of cartilage homeostasis via coupled thermal-pressure stimuli: a transcriptomic study in human cartilage explants](https://doi.org/10.3389/fmede.2023.1291191)
- 物理的な負荷 (hydrostatic pressure (HP))をかけているサンプル、段階的な温度上昇 (Isolated temperature increase group (“T”):) を加えているサンプル、複合的なサンプルなどが含まれている
- 負荷をかけていないサンプルとTグループのサンプルを収集

In [38]:
GSE242710_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE242710/GSE242710_PEP/GSE242710_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("NEBNext Ultra II Directional RNA Library Prep Kit").alias("rna_library"), # add rna_library column from GEO webpage (NEW ENGLAND Biolabs)
    pl.lit("https://doi.org/10.3389/fmede.2023.1291191").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("32.5 C to 38.7 C (gradient)").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("90 min. × 5 days").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("32.5 C").alias("control treatment temperature"), # add control temperature column
    pl.lit("cartilage").alias("cell_line"),
    pl.lit("chondrocyte").alias("cell_type")
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE242710_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE242710_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE242710_iseq_filtered = GSE242710_iseq[[s.name for s in GSE242710_iseq if not (s.null_count() == GSE242710_iseq.height)]]

GSE242710_combined = GSE242710_PEP.join(
    GSE242710_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE242710_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE242710_modified_metadata.tsv",
    separator='\t'
)

# display(GSE242710_combined)

In [39]:
defined_pairs_gse242710 = [
    ("SRR25954666", "SRR25954670"), # cartilage_isolated_t_stimulation_rep1 vs cartilage_unstimulated_rep1
    ("SRR25954665", "SRR25954669"), # cartilage_isolated_t_stimulation_rep2 vs cartilage_unstimulated_rep2
    ("SRR25954664", "SRR25954668"), # cartilage_combined_hp-t_stimulation_rep1 vs cartilage_isolated_hp_stimulation_rep1
    ("SRR25954663", "SRR25954667")  # cartilage_combined_hp-t_stimulation_rep2 vs cartilage_isolated_hp_stimulation_rep2
]

paired_metadata_df_gse242710 = create_paired_metadata(
    combined_df=GSE242710_combined,
    defined_sample_pairs=defined_pairs_gse242710,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse242710 is not None:
    paired_metadata_df_gse242710.write_csv(
        "../../Data/metadata/metadata_paired/GSE242710_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse242710)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,tissue,cell_type,treatment,assembly,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,cell_line,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,country,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias,tissue_type
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR25954666""","""SRR25954670""","""cartilage_isolated_t_stimulation_rep1 vs cartilage_unstimulated_rep1""","""null""","""null""","""null""","""null""","""null""","""null""","""cartilage, isolated T stimulation, rep1 vs cartilage, unstimulated, rep1""","""GSM7767455 vs GSM7767451""","""Public on Dec 01 2023""","""Sep 08 2023""","""Dec 01 2023""","""SRA""","""1""","""cartilage""","""Homo sapiens""","""9606""","""polyA RNA""","""T_1_S7_b38 vs C_1_S3_b38""","""GPL24676""","""Dominique,P.,Pioletti""","""LBO""","""STI IBI""","""EPFL""","""EPFL""","""Lausanne""","""Vaud""","""1015""","""Switzerland""","""Illumina NovaSeq 6000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM7767455 vs GSM7767451""","""NONE""","""GSE242710""","""0""","""SRX21673220 vs SRX21673216""","""cartilage""","""chondrocyte""","""isolated T stimulation vs unstimulated""","""GRCh38""",""".txt gene counts of 8 samples""","""SAMN37324517 vs SAMN37324521""","""NEBNext Ultra II Directional RNA Library Prep Kit""","""https://doi.org/10.3389/fmede.2023.1291191""","""32.5 C to 38.7 C (gradient)""","""90 min. × 5 days""","""32.5 C""","""cartilage""","""SRR25954666 vs SRR25954670""","""Illumina NovaSeq 6000 sequencing: GSM7767455: cartilage, isolated T stimulation, rep1 Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM7767451: cartilage, unstimulated, rep1 Homo sapiens R…","""Synergetic Support of Cartilage Homeostasis via Coupled Thermal-Pressure Stimuli: A Transcriptomic Study in Human Cartilage Explants""","""SRA1708025""","""cDNA""","""2024-03-27 vs 2023-12-13""","""712402011 vs 1426605452""","""597648984;603372864 vs 1005645641;1063006588""","""ILLUMINA""","""missing""","""Generic""","""cartilage, isolated T stimulation, rep1 vs cartilage, unstimulated, rep1""","""SRP459273""","""21549493 vs 43418885""","""Synergetic Support of Cartilage Homeostasis via Coupled Thermal-Pressure Stimuli: A Transcriptomic Study in Human Cartilage Explants""","""cartilage, isolated T stimulation, rep1 vs cartilage, unstimulated, rep1""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: GSM7767455: cartilage, isolated T stimul

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE182204 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-03
- [Acetyl-methyllysine marks chromatin at active transcription start sites](https://doi.org/10.1038/s41586-023-06565-9)
- ChIP-seqなども含まれており、これらはフィルタリングする
- RNA-seqに加え、TT-TL-seq (transient-transcriptome TimeLapse sequencing), start-TimeLapse sequencing (STL-seq) も含まれている
- 熱ストレス処理のデータは Drosophila melanogaster の ChIP-seqなので今回はこのデータは収集しない

In [40]:
# GSE182204_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE182204/GSE182204_PEP/GSE182204_PEP_raw.csv",
#     separator=','
# ).filter(
#     (pl.col("sample_organism_ch1") == "Homo sapiens") &
#     ~((pl.col("sample_library_strategy") == "ChIP-Seq") | (pl.col("sample_molecule_ch1") == "genomic DNA"))
# )

# display(GSE182204_PEP)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 14. GSE221579 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date: 2025-05-03
- [Transcriptional Response in Human Jurkat T Lymphocytes to a near Physiological Hypergravity Environment and to One Common in Routine Cell Culture Protocols](https://doi.org/10.3390/ijms24021351)
- hypergravity (hypg) などが含まれている
- 同じサンプル名の`Ctrl`と`heat`を選抜、解析に使用する
- 温度条件は42℃, 5分 ( `Materials & Methods` より)

In [ ]:
GSE221579_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE221579/GSE221579_PEP/GSE221579_PEP_raw.csv",
    separator=','
).filter(
    ~(pl.col("treatment").str.contains(r'hypg') | pl.col("treatment").str.contains(r'wait')) & 
    pl.col("sample_title").str.contains(r'-Achilles_')
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("E7765L NEBNext Ultra II Directional RNA Library Prep with Beads").alias("rna_library"), # add rna_library column from GEO webpage (SRX)
    pl.lit("https://doi.org/10.3390/ijms24021351").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("42 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("5 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE221579_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE221579_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE221579_iseq_filtered = GSE221579_iseq[[s.name for s in GSE221579_iseq if not (s.null_count() == GSE221579_iseq.height)]]

GSE221579_combined = GSE221579_PEP.join(
    GSE221579_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE221579_combined = GSE221579_combined[[s.name for s in GSE221579_combined if not (s.null_count() == GSE221579_combined.height)]]

GSE221579_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE221579_modified_metadata.tsv",
    separator='\t'
)

# display(GSE221579_combined)

In [42]:
defined_pairs_gse221579 = [
    ("SRR22865323", "SRR22865339"), # 65_65_gbf2021-achilles_a-heat-65 vs 49_49_gbf2021-achilles_a-ctrl-49
    ("SRR22865322", "SRR22865338"), # 66_66_gbf2021-achilles_a-heat-66 vs 50_50_gbf2021-achilles_a-ctrl-50
    ("SRR22865321", "SRR22865337"), # 67_67_gbf2021-achilles_a-heat-67 vs 51_51_gbf2021-achilles_a-ctrl-51
    ("SRR22865320", "SRR22865336")  # 68_68_gbf2021-achilles_a-heat-68 vs 52_52_gbf2021-achilles_a-ctrl-52
]

paired_metadata_df_gse221579 = create_paired_metadata(
    combined_df=GSE221579_combined,
    defined_sample_pairs=defined_pairs_gse221579,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse221579 is not None:
    paired_metadata_df_gse221579.write_csv(
        "../../Data/metadata/metadata_paired/GSE221579_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse221579)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,tissue,cell_line,cell_type,centrifuge,treatment,assembly,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias,tissue_type
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR22865323""","""SRR22865339""","""65_65_gbf2021-achilles_a-heat-65 vs 49_49_gbf2021-achilles_a-ctrl-49""","""65_65_GBF2021-Achilles_A-heat-65 vs 49_49_GBF2021-Achilles_A-Ctrl-49""","""GSM6886658 vs GSM6886642""","""Public on Feb 07 2023""","""Dec 22 2022""","""Feb 07 2023""","""SRA""","""1""","""in vitro immortalized cells""","""Homo sapiens""","""9606""","""total RNA""","""65_65_GBF2021-Achilles_A-heat-65_S65_R1_001_val_1.fqAligned.sortedByCoord.out.bam vs 49_49_GBF2021-Achilles_A-Ctrl-49_S49_R1_001_val_1.fqAligned.sortedByCoord.out.bam""","""GPL24676""","""Cora,S.,Thiel""","""cora.thiel@uzh.ch""","""Institute of Anatomy""","""University of Zurich""","""Winterthurerstrasse 190""","""Zurich""","""8057""","""Switzerland""","""Illumina NovaSeq 6000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM6886658 vs GSM6886642""","""NONE""","""GSE221579""","""0""","""SRX18824090 vs SRX18824074""","""in vitro immortalized cells""","""T lymphocyte""","""Jurkat T cells""","""GBF2021-Achilles""","""heat vs Ctrl""","""hg38""","""featureCounts output, raw counts in a table""","""SAMN32368461 vs SAMN32368477""","""E7765L NEBNext Ultra II Directional RNA Library Prep with Beads""","""https://doi.org/10.1038/s41586-023-06565-9""","""42 C""","""5 min.""","""37 C (inferred)""","""SRR22865323 vs SRR22865339""","""Illumina NovaSeq 6000 sequencing: GSM6886658: 65_65_GBF2021-Achilles_A-heat-65 Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM6886642: 49_49_GBF2021-Achilles_A-Ctrl-49 Homo sapiens RNA-S…","""Short-term hypergravity effects on Jurkat T cells""","""SRA1564384""","""cDNA""","""2023-02-11 vs 2023-02-10""","""3322278617 vs 3105935891""","""2458519226;2562037859 vs 2308142529;2398707865""","""ILLUMINA""","""Generic""","""65_65_GBF2021-Achilles_A-heat-65 vs 49_49_GBF2021-Achilles_A-Ctrl-49""","""SRP414464""","""46817903 vs 43982673""","""Short-term hypergravity effects on Jurkat T cells""","""65_65_GBF2021-Achilles_A-heat-65 vs 49_49_GBF2021-Achilles_A-Ctrl-49""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: GSM6886658: 65_65_GBF2021-Achilles_A-heat-65 Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM6886642: 49_49_GBF2021-Achilles_A-Ctrl-49 Homo sapiens RNA-S…","

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 15. GSE164834 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date: 2025-05-03
- [Evidence for mitochondrial Lonp1 expression in the nucleus](https://doi.org/10.1038/s41598-022-14860-0)
- Lonp1を標的としたRNAiなどのデータを含む37℃と42℃のデータを含む
- 全てのデータを解析に使用する

In [43]:
GSE164834_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE164834/GSE164834_PEP/GSE164834_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("TruSeq Stranded mRNA Library Prep Kit").alias("rna_library"), # add rna_library column from GEO webpage (Illumina) Methods に記載されている
    pl.lit("https://doi.org/10.1038/s41598-022-14860-0").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("42 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("60 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE164834_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE164834_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE164834_iseq_filtered = GSE164834_iseq[[s.name for s in GSE164834_iseq if not (s.null_count() == GSE164834_iseq.height)]]

GSE164834_combined = GSE164834_PEP.join(
    GSE164834_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE164834_combined = GSE164834_combined[[s.name for s in GSE164834_combined if not (s.null_count() == GSE164834_combined.height)]]

GSE164834_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE164834_modified_metadata.tsv",
    separator='\t'
)

# display(GSE164834_combined)

In [44]:
defined_pairs_gse164834 = [
    ("SRR13436390", "SRR13436384"), # HS_siCtrl_rep1 vs Ctrl_siCtrl_rep1
    ("SRR13436391", "SRR13436385"), # HS_siCtrl_rep2 vs Ctrl_siCtrl_rep2
    ("SRR13436392", "SRR13436386"), # HS_siCtrl_rep3 vs Ctrl_siCtrl_rep3
    ("SRR13436393", "SRR13436387"), # HS_siLONP1_rep1 vs Ctrl_siLONP1_rep1
    ("SRR13436394", "SRR13436388"), # HS_siLONP1_rep2 vs Ctrl_siLONP1_rep2
    ("SRR13436395", "SRR13436389"), # HS_siLONP1_rep3 vs Ctrl_siLONP1_rep3
    # ("SRR13436399", "SRR13436396"), # HS_REC_siLONP1_rep1 vs HS_REC_siCtrl_rep1
    # ("SRR13436400", "SRR13436397"), # HS_REC_siLONP1_rep2 vs HS_REC_siCtrl_rep2
    # ("SRR13436401", "SRR13436398")  # HS_REC_siLONP1_rep3 vs HS_REC_siCtrl_rep3
]

paired_metadata_df_gse164834 = create_paired_metadata(
    combined_df=GSE164834_combined,
    defined_sample_pairs=defined_pairs_gse164834,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse164834 is not None:
    paired_metadata_df_gse164834.write_csv(
        "../../Data/metadata/metadata_paired/GSE164834_paired_metadata.tsv",
        separator='\t'
    )
    pl.Config.set_tbl_rows(2)
    display(paired_metadata_df_gse164834)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,treatment,condition,cell_line,genome_build,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR13436390""","""SRR13436384""","""hs_sictrl_rep1 vs ctrl_sictrl_rep1""","""cDNA""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR13436390 vs SRR13436384""","""SRX9849901 vs SRX9849895""","""HS_siCtrl_rep1 vs Ctrl_siCtrl_rep1""","""GSM5020469 vs GSM5020463""","""Public on Jun 28 2022""","""Jan 14 2021""","""Jun 28 2022""","""SRA""","""1""","""Colon cancer cells""","""Homo sapiens""","""9606""","""total RNA""","""GPL24676""","""Mattia,,Forcato""","""University of Padova""","""Via Ugo Bassi, 58/B""","""Padova""","""35131""","""Italy""","""Illumina NovaSeq 6000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM5020469 vs GSM5020463""","""NONE""","""GSE164834""","""0""","""SRX9849901 vs SRX9849895""","""Scrambled siRNA""","""Kept at 42°C for one hour vs Kept at 37°C""","""SW620""","""hg38""","""SAMN17315583 vs SAMN17315586""","""TruSeq Stranded mRNA Library Prep Kit""","""https://doi.org/10.1038/s41598-022-14860-0""","""42 C""","""60 min.""","""37 C""","""SRR13436390 vs SRR13436384""","""Illumina NovaSeq 6000 sequencing: GSM5020469: HS_siCtrl_rep1 Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM5020463: Ctrl_siCtrl_rep1 Homo sapiens RNA-Seq""","""Mitochondrial protease Lonp1 localizes to the nucleus in response to heat shock""","""SRA1183843""","""cDNA""","""2022-06-30""","""3455709475 vs 3166303408""","""2496252966;2641010718 vs 2282923311;2425160175""","""ILLUMINA""","""Generic""","""HS_siCtrl_rep1 vs Ctrl_siCtrl_rep1""","""SRP301760""","""55272865 vs 50316726""","""Mitochondrial protease Lonp1 localizes to the nucleus in response to heat shock""","""HS_siCtrl_rep1 vs Ctrl_siCtrl_rep1""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: GSM5020469: HS_siCtrl_rep1 Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM5020463: Ctrl_siCtrl_rep1 Homo sapiens RNA-Seq""","""11038685130 vs 10075832350""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""public""","""GSM5020469 vs GSM5020463""","""GSM5020469_r1 vs GSM5020463_r1""","""RNA was extracted by using RNeasy minikit following manufacturer's instructions. Libraries were prepared usinn TruSeq™ Stranded mRNA e Total RNA according to manufacturer's instructions""","""GSM5020469 vs GSM5020463""","""GEO""","""SW620""","""2022-06-30""","""PRJNA692181""","""SRS8031417 vs SRS8031411""","""2022-06-30""","""GSE164834"""
…,…,…,

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE168716 (PAIRED) NO DOWNLOAD

- validate date: 2025-05-03
- [Internally controlled RNA sequencing comparisons using nucleoside recoding chemistry](https://doi.org/10.1093/nar/gkac693)
- ヒトのデータとDrosophilaのデータが含まれており、このうち、熱ストレス処理をしているデータはDrosophilaだったので、このデータは今回は使用しない

In [45]:
# GSE168716_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE168716/GSE168716_PEP/GSE168716_PEP_raw.csv",
#     separator=','
# ).filter(
#     pl.col("sample_organism_ch1") == "Homo sapiens"
# )

# display(GSE168716_PEP)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 16. GSE182960 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date: 2025-05-04
- [Cohesin regulates alternative splicing](https://doi.org/10.1126/sciadv.ade3876)
- オーキシン処理によりRAD21の分解が行われ、コヒーシンの機能が失われるHCT116RmAC cellsが用いられている研究
- `Heat shock treatment` セクションに詳細が書いてある
- `treatment`を見ると、熱ショックを加えているものとそうでないサンプルがわかりやすく明記されている
- DMSO処理、タンパク質分解誘導キメラタンパク質であるPROTAC MZ1処理、MZ1+インドール酢酸(IAA)処理、1× phosphate-buffered saline (PBS)処理、IAA処理のサンプルはフィルタリングする
- 熱ショック + MZ1などの処理も含めて解析に使用する

In [46]:
ID = "GSE182960"

GSE182960_PEP = pl.read_csv(
    f"../../Data/metadata/metadata_geofetch/{ID}/{ID}_PEP/{ID}_PEP_raw.csv",
    separator=','
).filter(
    ~((pl.col("treatment") == "DMSO") | (pl.col("treatment") == "MZ1") | (pl.col("treatment") == "MZ1+IAA") | (pl.col("treatment") == "PBS") | (pl.col("treatment") == "IAA"))
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("TruSeq Stranded mRNA Library").alias("rna_library"), # add rna_library column from GEO webpage (Illumina)
    pl.lit("https://doi.org/10.1126/sciadv.ade3876").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("42 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("60 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE182960_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE182960_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE182960_iseq_filtered = GSE182960_iseq[[s.name for s in GSE182960_iseq if not (s.null_count() == GSE182960_iseq.height)]]

GSE182960_combined = GSE182960_PEP.join(
    GSE182960_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE182960_combined = GSE182960_combined[[s.name for s in GSE182960_combined if not (s.null_count() == GSE182960_combined.height)]]

GSE182960_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE182960_modified_metadata.tsv",
    separator='\t'
)

# display(GSE182960_combined)

In [47]:
defined_pairs_gse182960 = [
    ("SRR15658378", "SRR15658366"), # 13_DMSOHSR1 vs 1_DMSONHSR1
    ("SRR15658379", "SRR15658367"), # 14_DMSOHSR2 vs 2_DMSONHSR2
    ("SRR15658380", "SRR15658368"), # 15_DMSOHSR3 vs 3_DMSONHSR3
    ("SRR15658381", "SRR15658369"), # 16_MZ1HSR1 vs 4_MZ1NHSR1
    ("SRR15658382", "SRR15658370"), # 17_MZ1HSR2 vs 5_MZ1NHSR2
    ("SRR15658383", "SRR15658371"), # 18_MZ1HSR3 vs 6_MZ1NHSR3
    ("SRR15658384", "SRR15658372"), # 19_IAAHSR1 vs 7_IAANHSR1
    ("SRR15658385", "SRR15658373"), # 20_IAAHSR2 vs 8_IAANHSR2
    ("SRR15658386", "SRR15658374"), # 21_IAAHSR3 vs 9_IAANHSR3
    ("SRR15658387", "SRR15658375"), # 22_MZ1_IAAHSR1 vs 10_MZ1_IAANHSR1
    ("SRR15658388", "SRR15658376"), # 23_MZ1_IAAHSR2 vs 11_MZ1_IAANHSR2
    ("SRR15658389", "SRR15658377")  # 24_MZ1_IAAHSR3 vs 12_MZ1_IAANHSR3
]

paired_metadata_df_gse182960 = create_paired_metadata(
    combined_df=GSE182960_combined, 
    defined_sample_pairs=defined_pairs_gse182960,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse182960 is not None:
    paired_metadata_df_gse182960.write_csv(
        "../../Data/metadata/metadata_paired/GSE182960_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse182960)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_type,treatment,genome_build,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR15658378""","""SRR15658366""","""13_dmsohsr1 vs 1_dmsonhsr1""","""cDNA""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR15658378 vs SRR15658366""","""SRX11955299 vs SRX11955287""","""13_DMSOHSR1 vs 1_DMSONHSR1""","""GSM5543754 vs GSM5543742""","""Public on Aug 31 2022""","""Aug 27 2021""","""Aug 31 2022""","""SRA""","""1""","""HCT116RmAC_DMSO under heat shock conditions vs HCT116RmAC_DMSO under non heat shock conditions""","""Homo sapiens""","""9606""","""total RNA""","""HCT116RmAC cells treated with DMSO under heat shock conditions replicate 1 vs HCT116RmAC cells treated with DMSO under non heat shock conditions replicate 1""","""GPL24676""","""Dinah,S.,Singer""","""dinah.singer@nih.gov""","""301-496-9097""","""Division of Cancer Biology""","""National Cancer Institute""","""Building 10, Room 4B-17""","""Bethesda""","""MD""","""20892""","""USA""","""Illumina NovaSeq 6000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM5543754 vs GSM5543742""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5543nnn/GSM5543754/suppl/GSM5543754_13_DMSOHSR1.htseq_count.tsv.gz vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5543nnn/GSM5543742/suppl/GSM5543742_1_DMSONHSR…","""GSE182960""","""0""","""SRX11955299 vs SRX11955287""","""HCT116RmAC""","""human colorectal carcinoma cell line""","""DMSO under heat shock conditions vs DMSO under non heat shock conditions""","""hg19""","""raw counts of sequencing reads for each sample""","""SAMN21027102 vs SAMN21027096""","""TruSeq Stranded mRNA Library""","""https://doi.org/10.1126/sciadv.ade3876""","""42 C""","""60 min.""","""37 C""","""SRR15658378 vs SRR15658366""","""Illumina NovaSeq 6000 sequencing: GSM5543754: 13_DMSOHSR1 Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM5543742: 1_DMSONHSR1 Homo sapiens RNA-Seq""","""Cohesin regulates alternative splicing alone or with BRD4""","""SRA1285104""","""cDNA""","""2022-11-19""","""3896989375 vs 3483905044""","""2771548070;2983285269 vs 2483568874;2667071170""","""ILLUMINA""","""Generic""","""13_DMSOHSR1 vs 1_DMSONHSR1""","""SRP334521""","""42049482 vs 37553413""","""Cohesin regulates alternative splicing alone or with BRD4""","""

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 17. GSE198441 (PAIRED) (MSR-seq)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date: 2025-05-04
- [A multiplex platform for small RNA sequencing elucidates multifaceted tRNA stress response and translational regulation](https://doi.org/10.1038/s41467-022-30261-3)
- small RNA-seqの手法に関する論文
- シュードウリジン付加のためのCMC(well-established N-cyclohexyl-N′-β-(4-methylmorpholinium) ethylcarbodiimide (CMC) reaction) 処理が行われているサンプル、過酸化水素、ヒ素処理などのサンプルを含む
- `_nostress_` と `_heatstress_`を含むデータを今回は収集
- `HEK293T_totalRNA_nostress_shortRT_minusDM_` と `HEK293T_totalRNA_nostress_shortRT_plusDM_` は対応する熱ストレスのデータがないことが判明したため、今回はフィルタリングする
- 温度条件は42℃,1時間 (論文より)
- データ解析手法は、通常の RNA-seq と ほぼ変わらないと判断したため、収集する
- ライブラリー調製の部分

In [48]:
ID = "GSE198441"

GSE198441_PEP = pl.read_csv(
    f"../../Data/metadata/metadata_geofetch/{ID}/{ID}_PEP/{ID}_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_title").str.contains(r'_nostress_') | 
     pl.col("sample_title").str.contains(r'_heatstress_')) & # filter out samples without stress
     
    ~((pl.col("sample_title").str.contains(r'HEK293T_totalRNA_nostress_shortRT_minusDM_')) | 
      (pl.col("sample_title").str.contains(r'HEK293T_totalRNA_nostress_shortRT_plusDM_'))) # filter out samples without stress
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("MSR-seq (custom mRNA library preparation)").alias("rna_library"), # add rna_library column from GEO webpage
    pl.lit("https://doi.org/10.1038/s41467-022-30261-3").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("42 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("60 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature"), # add control temperature column
    pl.lit("HEK293T").alias("cell_line")
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE198441_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE198441_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE198441_iseq_filtered = GSE198441_iseq[[s.name for s in GSE198441_iseq if not (s.null_count() == GSE198441_iseq.height)]]

GSE198441_combined = GSE198441_PEP.join(
    GSE198441_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE198441_combined = GSE198441_combined[[s.name for s in GSE198441_combined if not (s.null_count() == GSE198441_combined.height)]]

GSE198441_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE198441_modified_metadata.tsv",
    separator='\t'
)

# display(GSE198441_combined)

In [49]:
defined_pairs_gse198441 = [
    ("SRR18302276", "SRR18302279"), # HEK293T_totalRNA_heatstress_longRT_minusDM_rep1 vs HEK293T_totalRNA_nostress_longRT_minusDM_rep1
    ("SRR18302275", "SRR18302278"), # HEK293T_totalRNA_heatstress_longRT_minusDM_rep2 vs HEK293T_totalRNA_nostress_longRT_minusDM_rep2
    ("SRR18302274", "SRR18302277"), # HEK293T_totalRNA_heatstress_longRT_minusDM_rep3 vs HEK293T_totalRNA_nostress_longRT_minusDM_rep3
    ("SRR18302264", "SRR18302267"), # HEK293T_polysomeRNA_heatstress_longRT_minusDM_rep1 vs HEK293T_polysomeRNA_nostress_longRT_minusDM_rep1
    ("SRR18302263", "SRR18302266"), # HEK293T_polysomeRNA_heatstress_longRT_minusDM_rep2 vs HEK293T_polysomeRNA_nostress_longRT_minusDM_rep2
    ("SRR18302262", "SRR18302265"), # HEK293T_polysomeRNA_heatstress_longRT_minusDM_rep3 vs HEK293T_polysomeRNA_nostress_longRT_minusDM_rep3
    ("SRR18302254", "SRR18302255"), # HEK293T_polyARNA_heatstress_longRT_minusDM_rep1 vs HEK293T_polyARNA_nostress_longRT_minusDM_rep1
    ("SRR18302252", "SRR18302253"), # HEK293T_polyARNA_heatstress_longRT_minusDM_rep2 vs HEK293T_polyARNA_nostress_longRT_minusDM_rep2
    ("SRR18302251", "SRR18302247"), # HEK293T_polyARNA_heatstress_longRT_minusDM_rep3 vs HEK293T_polyARNA_nostress_longRT_minusDM_rep3
    ("SRR18302240", "SRR18302243"), # HEK293T_polysomepolyARNA_heatstress_longRT_minusDM_rep1 vs HEK293T_polysomepolyARNA_nostress_longRT_minusDM_rep1
    ("SRR18302238", "SRR18302242"), # HEK293T_polysomepolyARNA_heatstress_longRT_minusDM_rep2 vs HEK293T_polysomepolyARNA_nostress_longRT_minusDM_rep2
    ("SRR18302239", "SRR18302241")  # HEK293T_polysomepolyARNA_heatstress_longRT_minusDM_rep3 vs HEK293T_polysomepolyARNA_nostress_longRT_minusDM_rep3
]

paired_metadata_df_gse198441 = create_paired_metadata(
    combined_df=GSE198441_combined,
    defined_sample_pairs=defined_pairs_gse198441,
    sample_id_column_name="accession"
)


if paired_metadata_df_gse198441 is not None:
    paired_metadata_df_gse198441.write_csv(
        "../../Data/metadata/metadata_paired/GSE198441_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse198441)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,strain,fraction,enzyme_chemical_treatment,rt_condition,assembly,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,cell_line,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,library_name,first_created,strain_right,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR18302276""","""SRR18302279""","""hek293t_totalrna_heatstress_longrt_minusdm_rep1 vs hek293t_totalrna_nostress_longrt_minusdm_rep1""","""HEK293T_totalRNA_heatstress_longRT_minusDM_rep1 vs HEK293T_totalRNA_nostress_longRT_minusDM_rep1""","""GSM5946922 vs GSM5946919""","""Public on Mar 13 2022""","""Mar 11 2022""","""Mar 27 2022""","""SRA""","""1""","""RNA extracted from HEK293T cell culture, after heat stress vs RNA extracted from HEK293T cell culture, with no stress treatment""","""Homo sapiens""","""9606""","""total RNA""","""GPL24676""","""Christopher,D,Katanski""","""katanski@uchicago.edu""","""7348297916""","""Pan lab""","""Biochemistry""","""The University of Chicago""","""1649 E. 50th St.""","""Chicago""","""Illinois""","""60615""","""USA""","""Illumina NovaSeq 6000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM5946922 vs GSM5946919""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5946nnn/GSM5946922/suppl/GSM5946922_basewise_HEK293T_totalRNA_heatstress_longRT_minusDM_rep1.tsv.gz vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5946nnn/GSM59…","""GSE198441""","""0""","""SRX14439722 vs SRX14439719""","""HEK293T""","""Total RNA""","""none""","""Overnight RT""","""Hg19; hg38""","""SAMN26583138 vs SAMN26583141""","""MSR-seq (custom mRNA library preparation)""","""https://doi.org/10.1038/s41467-022-30261-3""","""42 C""","""60 min.""","""37 C (inferred)""","""HEK293T""","""SRR18302276 vs SRR18302279""","""Illumina NovaSeq 6000 sequencing: GSM5946922: HEK293T_totalRNA_heatstress_longRT_minusDM_rep1 Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM5946919: HEK293T_totalRNA_nostress_longRT_min…","""A multiplex platform for small RNA sequencing elucidates multifaceted tRNA stress response and translational regulation""","""SRA1383949""","""cDNA""","""2022-04-03""","""2450785308 vs 3183003023""","""1042026536;1040587098 vs 1371261579;1358915183""","""ILLUMINA""","""Generic""","""HEK293T_totalRNA_heatstress_longRT_minusDM_rep1 vs HEK293T_totalRNA_nostress_longRT_minusDM_rep1""","""SRP363629""","""25845765 vs 33389609""","""A multiplex platform for small RNA sequencing elucidates multifaceted tRNA stress response and translational regulation""","""HEK293T_totalRNA_heatstress_longRT_minusDM_rep1 vs HE

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 18. GSE164846 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date: 2025-05-04
- [Local translation in nuclear condensate amyloid bodies](https://doi.org/10.1073/pnas.2014457118)
- おそらく上記の論文由来のデータ(GEOのページには登録されていなかった)
- `Basal: 37°C, 21% O2, pH 7.4. Heat shock: 43°C, 21% O2, pH 7.4`
- 全てのデータを収集する

In [50]:
ID = "GSE164846"

GSE164846_PEP = pl.read_csv(
    f"../../Data/metadata/metadata_geofetch/{ID}/{ID}_PEP/{ID}_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("KAPA Stranded mRNA-Seq Kit").alias("rna_library"), # add rna_library column from GEO webpage (KAPA Biosystems)
    pl.lit("https://doi.org/10.1073/pnas.2014457118").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("43 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("60 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE164846_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE164846_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE164846_iseq_filtered = GSE164846_iseq[[s.name for s in GSE164846_iseq if not (s.null_count() == GSE164846_iseq.height)]]

GSE164846_combined = GSE164846_PEP.join(
    GSE164846_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE164846_combined = GSE164846_combined[[s.name for s in GSE164846_combined if not (s.null_count() == GSE164846_combined.height)]]

GSE164846_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE164846_modified_metadata.tsv",
    separator='\t'
)

# display(GSE164846_combined)

In [51]:
defined_pairs_gse164846 = [
    ("SRR13436541", "SRR13436538"), # Purified Amyloid-bodies following 60min heat shock 1 vs Total RNA from MCF7 cells, purified nucleoli from cells maintained at basal conditions 1
    ("SRR13436542", "SRR13436539"), # Purified Amyloid-bodies following 60min heat shock 2 vs Total RNA from MCF7 cells, purified nucleoli from cells maintained at basal conditions 2
    ("SRR13436543", "SRR13436540")  # Purified Amyloid-bodies following 60min heat shock 3 vs Total RNA from MCF7 cells, purified nucleoli from cells maintained at basal conditions 3
]

paired_metadata_df_gse164846 = create_paired_metadata(
    combined_df=GSE164846_combined,
    defined_sample_pairs=defined_pairs_gse164846,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse164846 is not None:
    paired_metadata_df_gse164846.write_csv(
        "../../Data/metadata/metadata_paired/GSE164846_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse164846)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,genome_build,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR13436541""","""SRR13436538""","""purified_amyloid-bodies_following_60min_heat_shock_1 vs total_rna_from_mcf7_cells_purified_nucleoli_from_cells_maintained_at_basal_conditions_1""","""cDNA""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR13436541 vs SRR13436538""","""SRX9850050 vs SRX9850053""","""Purified Amyloid-bodies following 60min heat shock 1 vs Total RNA from MCF7 cells, purified nucleoli from cells maintained at basal conditions 1""","""GSM5020921 vs GSM5020918""","""Public on Jan 15 2021""","""Jan 14 2021""","""Jan 17 2021""","""SRA""","""1""","""MCF7 mammary gland cell culture""","""Homo sapiens""","""9606""","""total RNA""","""adenocarcinoma""","""GPL18573""","""Stephen,,Lee""","""stephenlee@med.miami.edu""","""Gautier Bldg, Rm 228""","""Biochemistry & Molecular Biology""","""University of Miami""","""1011 NW 15 st.""","""Miami""","""Florida""","""33136""","""USA""","""Illumina NextSeq 500""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM5020921 vs GSM5020918""","""NONE""","""GSE164846""","""0""","""SRX9850050 vs SRX9850053""","""MCF7""","""GRCh38 (gencode v27)""","""SAMN17315995 vs SAMN17315998""","""KAPA Stranded mRNA-Seq Kit""","""https://doi.org/10.1073/pnas.2014457118""","""43 C""","""60 min.""","""37 C""","""SRR13436541 vs SRR13436538""","""NextSeq 500 sequencing: GSM5020921: Purified Amyloid-bodies following 60min heat shock 1 Homo sapiens RNA-Seq vs NextSeq 500 sequencing: GSM5020918: Total RNA from MCF7 cells, purified nucleoli from c…","""Local Translation in Nuclear Condensate Amyloid-bodies""","""SRA1183887""","""cDNA""","""2021-01-19""","""2161746319 vs 2204816307""","""1390794867;1413239310 vs 1436805689;1464612808""","""ILLUMINA""","""Generic""","""Purified Amyloid-bodies following 60min heat shock 1 vs Total RNA from MCF7 cells, purified nucleoli from cells maintained at basal conditions 1""","""SRP301773""","""17858977 vs 18347116""","""Local Translation in Nuclear Condensate Amyloid-bodies""","""Purified Amyloid-bodies following 60min heat shock 1 vs Total RNA from MCF7 cells, purified nucleoli from cells maintained at basal conditions 1""","""NextSeq 500""","""NextSeq 500 sequencing: GSM5020921: Purified Amyloid-bodies following 60min heat shock 1 Homo sapien

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 19. GSE149797 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date: 2025-05-04
- [Proteasome stress in skeletal muscle mounts a long-range protective response that delays retinal and brain aging](https://doi.org/10.1016/j.cmet.2021.03.005)
- HS + maltose, HSのみの処理が含まれている
- 以前の研究でも使用したデータ
- 今回も全てのデータを使用する
- おそらくこのデータは41.5℃で7時間

In [52]:
ID = "GSE149797"

GSE149797_PEP = pl.read_csv(
    f"../../Data/metadata/metadata_geofetch/{ID}/{ID}_PEP/{ID}_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("TruSeq stranded mRNA library preparation kit").alias("rna_library"), # add rna_library column from Article (Illumina)
    pl.lit("https://doi.org/10.1016/j.cmet.2021.03.005").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("41.5 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("420 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature"), # add control temperature column
    pl.lit("cortical brain organoids").alias("cell_line")
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE149797_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE149797_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE149797_iseq_filtered = GSE149797_iseq[[s.name for s in GSE149797_iseq if not (s.null_count() == GSE149797_iseq.height)]]

GSE149797_combined = GSE149797_PEP.join(
    GSE149797_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE149797_combined = GSE149797_combined[[s.name for s in GSE149797_combined if not (s.null_count() == GSE149797_combined.height)]]

GSE149797_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE149797_modified_metadata.tsv",
    separator='\t'
)

# display(GSE149797_combined)

In [53]:
defined_pairs_gse149797 = [
    ("SRR11684069", "SRR11684066"), # HS_control_1 vs control_1
    ("SRR11684070", "SRR11684067"), # HS_control_2 vs control_2
    ("SRR11684071", "SRR11684068"), # HS_control_3 vs control_3
    ("SRR11684072", "SRR11684075"), # HS_maltose_1 vs maltose_1
    ("SRR11684073", "SRR11684076"), # HS_maltose_2 vs maltose_2
    ("SRR11684074", "SRR11684077")  # HS_maltose_3 vs maltose_3
]

paired_metadata_df_gse149797 = create_paired_metadata(
    combined_df=GSE149797_combined,
    defined_sample_pairs=defined_pairs_gse149797,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse149797 is not None:
    paired_metadata_df_gse149797.write_csv(
        "../../Data/metadata/metadata_paired/GSE149797_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse149797)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_characteristics_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,genome_build,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,cell_line,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias,tissue_type
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR11684069""","""SRR11684066""","""hs_control_1 vs control_1""","""cDNA""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR11684069 vs SRR11684066""","""SRX8244864 vs SRX8244861""","""HS_control_1 vs control_1""","""GSM4512824 vs GSM4512821""","""Public on Feb 12 2021""","""May 04 2020""","""Feb 12 2021""","""SRA""","""1""","""control with heat shock replicate 1 vs control without heat shock replicate 1""","""Homo sapiens""","""9606""","""tissue: cortical brain organoids""","""total RNA""","""control with heat shock replicate 1 vs control without heat shock replicate 1""","""GPL11154""","""Yong-Dong,,Wang""","""michael.wang@stjude.org""","""9015954224""","""St. Jude Children's Research Hospital""","""262 Danny Thomas Pl, MS1135""","""Memphis""","""TN""","""38134""","""USA""","""Illumina HiSeq 2000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM4512824 vs GSM4512821""","""NONE""","""GSE149797, GSE149799""","""0""","""SRX8244864 vs SRX8244861""","""hg38""","""SAMN14827271 vs SAMN14827262""","""TruSeq stranded mRNA library preparation kit""","""https://doi.org/10.1016/j.cmet.2021.03.005""","""41.5 C""","""420 min.""","""37 C (inferred)""","""cortical brain organoids""","""SRR11684069 vs SRR11684066""","""Illumina HiSeq 2000 sequencing: GSM4512824: HS_control_1 Homo sapiens RNA-Seq vs Illumina HiSeq 2000 sequencing: GSM4512821: control_1 Homo sapiens RNA-Seq""","""Proteasome Stress in Skeletal Muscle Mounts a Long-Range Protective Response that Delays Retinal and Brain Aging [Human]""","""SRA1072146""","""cDNA""","""2021-02-14""","""3111185835 vs 3193202064""","""2322103403;2408757282 vs 2424029701;2513330211""","""ILLUMINA""","""Generic""","""HS_control_1 vs control_1""","""SRP259995""","""50877637 vs 52520376""","""Proteasome Stress in Skeletal Muscle Mounts a Long-Range Protective Response that Delays Retinal and Brain Aging [Human]""","""HS_control_1 vs control_1""","""Illumina HiSeq 2000""","""Illumina HiSeq 2000 sequencing: GSM4512824: HS_control_1 Homo sapiens RNA-Seq vs Illumina HiSeq 2000 sequencing: GSM4512821: control_1 Homo sapiens RNA-Seq""","""10277282674 vs 10609115952""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""public""","""GSM4512824 vs GSM4512821""","""GSM4512824_r1 vs GSM45

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE154746 (PAIRED) PRO-seq

- validate date: 2025-05-04
- [Stress-induced transcriptional memory accelerates promoter-proximal pause release and decelerates termination over mitotic divisions](https://doi.org/10.1016/j.molcel.2021.03.007)
- [実験医学: PRO-seq](https://www.yodosha.co.jp/jikkenigaku/keyword/PRO-seq/id/107099)
- `unconditioned (uC) or preconditioned with a single heat shock exposure (pC)` という条件がある
- 42℃で複数の曝露時間条件が含まれている
- 処理を[12_7_metadata_proseq.ipynb](./12_7_metadata_proseq.ipynb)に移動

### GSE134813 (PAIRED) NO DOWNLOAD

- validate date: 2025-05-04
- [The Alazami Syndrome-Associated Protein LARP7 Guides U6 Small Nuclear RNA Modification and Contributes to Splicing Robustness](https://doi.org/10.1016/j.molcel.2020.01.001)
- LARP7に関する研究
- コントロールとなるデータが含まれていなかったので今回はこのデータは収集しない

In [54]:
# ID = "GSE134813"

# GSE134813_PEP = pl.read_csv(
#     f"../../Data/metadata/metadata_geofetch/{ID}/{ID}_PEP/{ID}_PEP_raw.csv",
#     separator=','
# )

# pl.Config.set_fmt_str_lengths(200)
# display(GSE134813_PEP)

### GSE128365 (PAIRED) scEdU-seq

- validate date: 2025-05-04
- [Quantifying DNA replication speeds in single cells by scEdU-seq](https://doi.org/10.1038/s41592-024-02308-4)
- `single-cell 5-ethynyl-2′-deoxyuridine sequencing (scEdU-seq)` という手法に関するデータ
- nacsent RNAの検出ができる手法
- ~~一応 scRNA-seqの亜種なので、今回は収集しない (もしかしたら後ほど収集するかも？)~~
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に処理を移動

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 20. GSE123980 (PAIRED) RNA-seq

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date: 2025-05-04
- [The pause-initiation limit restricts transcription activation in human cells](https://doi.org/10.1038/s41467-019-11536-8)
- TT-seq,RNA-seq,mNET-seqなどの複数の手法が含まれている
- [TT-seqに関する論文: TT-seq maps the human transient transcriptome](https://doi.org/10.1126/science.aad9841)
- [mNET-seq](https://jp.illumina.com/science/sequencing-method-explorer/kits-and-arrays/mnet-seq.html)
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)にbulk RNA-seq以外のデータの処理を移行

In [55]:
ID = "GSE123980"

GSE123980_PEP = pl.read_csv(
    f"../../Data/metadata/metadata_geofetch/{ID}/{ID}_PEP/{ID}_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("Ovation Universal RNA-Seq System").alias("rna_library"), # add rna_library column from GEO webpage (NuGEN)
    pl.lit("https://doi.org/10.1038/s41467-019-11536-8").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("42 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("37 C").alias("control treatment temperature"), # add control temperature column


    pl.when(
        pl.col("sample_name").str.contains(r'_hs15_')
    ).then(
        pl.lit("15 min.")
    ).when(
        pl.col("sample_name").str.contains(r'_hs30_')
    ).then(
        pl.lit("30 min.")
    ).otherwise(
        pl.lit(None)
    ).alias("stress treatment time")
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

GSE123980_PEP_rna_seq = GSE123980_PEP.filter(
    pl.col("sample_title").str.contains(r'RNA-seq_')
)

# metadata from iseq
GSE123980_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE123980_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE123980_iseq_filtered = GSE123980_iseq[[s.name for s in GSE123980_iseq if not (s.null_count() == GSE123980_iseq.height)]]

GSE123980_rna_seq_combined = GSE123980_PEP_rna_seq.join(
    GSE123980_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE123980_rna_seq_combined = GSE123980_rna_seq_combined[[s.name for s in GSE123980_rna_seq_combined if not (s.null_count() == GSE123980_rna_seq_combined.height)]]

GSE123980_rna_seq_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE123980_rna_seq_modified_metadata.tsv",
    separator='\t'
)

# pl.Config.set_tbl_rows(100)
# display(GSE123980_rna_seq_combined)

In [56]:
# RNA-seq
defined_pairs_gse123980_rna_seq = [
    ("SRR8334021", "SRR8334019"), # RNA-seq_K562_HS15_rep1 vs RNA-seq_K562_Ctrl_rep1
    ("SRR8334022", "SRR8334020"), # RNA-seq_K562_HS15_rep2 vs RNA-seq_K562_Ctrl_rep2
    ("SRR8334023", "SRR8334019"), # RNA-seq_K562_HS30__rep1 vs RNA-seq_K562_Ctrl_rep1
    ("SRR8334024", "SRR8334020")  # RNA-seq_K562_HS30_rep2 vs RNA-seq_K562_Ctrl_rep2
]

paired_metadata_df_gse123980_rna_seq = create_paired_metadata(
    combined_df=GSE123980_rna_seq_combined,
    defined_sample_pairs=defined_pairs_gse123980_rna_seq,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse123980_rna_seq is not None:
    # Strip residual "null vs " prefix and " vs null" suffix from all string columns,
    # which appear when only one side of a pair has a value.
    paired_metadata_df_gse123980_rna_seq = paired_metadata_df_gse123980_rna_seq.with_columns(
        pl.col(pl.String)
        .str.replace_all(r"^null vs ", "") # control treatment temperature column から "null vs " を削除
        .str.replace_all(r" vs null$", "") # stress treatment temperature column から "vs null" を削除
    )
    # Control samples have no stress time, so "15 min. vs null" became null.
    # Set the heat-side time from sample_title for each pair.
    paired_metadata_df_gse123980_rna_seq = paired_metadata_df_gse123980_rna_seq.with_columns(
        pl.when(
            pl.col("sample_title").str.contains(r'^RNA-seq_K562_HS15_')
        ).then(
            pl.lit("15 min.")
        ).when(
            pl.col("sample_title").str.contains(r'^RNA-seq_K562_HS30_')
        ).then(
            pl.lit("30 min.")
        ).otherwise(
            pl.col("stress treatment time")
        ).alias("stress treatment time")
    )
    pl.Config.set_tbl_rows(2)
    display(paired_metadata_df_gse123980_rna_seq)
    paired_metadata_df_gse123980_rna_seq.write_csv(
        "../../Data/metadata/metadata_paired/GSE123980_rna_seq_paired_metadata.tsv",
        separator='\t'
    )
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_supplementary_file_2,gse_id,sample_data_row_count,experiment_accession,cell_line,treatment_conditions,molecule,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,control treatment temperature,stress treatment time,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR8334021""","""SRR8334019""","""rna-seq_k562_hs15_rep1 vs rna-seq_k562_ctrl_rep1""","""cDNA""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR8334021 vs SRR8334019""","""SRX5145539 vs SRX5145537""","""RNA-seq_K562_HS15_rep1 vs RNA-seq_K562_Ctrl_rep1""","""GSM3518113 vs GSM3518111""","""Public on Jul 10 2019""","""Dec 17 2018""","""Jul 10 2019""","""SRA""","""1""","""K562 erythroleukemia cells""","""Homo sapiens""","""9606""","""total RNA""","""Ovation Universal RNA-Seq System (NuGEN), TT-seq, RNA-seq (Schwalb et al., 2016), or mNET-seq (Nojima et al., 2016; Schlackow et al., 2017).""","""GPL18573""","""Björn,,Schwalb""","""bschwal@gwdg.de""","""MPI""","""Fassberg 11""","""Göttingen""","""37077""","""Germany""","""Illumina NextSeq 500""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM3518113 vs GSM3518111""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3518nnn/GSM3518113/suppl/GSM3518113_T_with_1_AGTGAG_3_K562_HS15.minus.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3518nnn/GSM3518111/suppl/GSM3518111_T_wi…","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3518nnn/GSM3518113/suppl/GSM3518113_T_with_1_AGTGAG_3_K562_HS15.plus.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3518nnn/GSM3518111/suppl/GSM3518111_T_wit…","""GSE123980""","""0""","""SRX5145539 vs SRX5145537""","""K562""","""4sU (Sigma) labeling for 5 minutes.""","""Total fragmented RNA""","""bigwig files containing the coverage""","""SAMN10604688 vs SAMN10604690""","""Ovation Universal RNA-Seq System""","""https://doi.org/10.1038/s41467-019-11536-8""","""42 C""","""37 C""","""15 min.""","""SRR8334021 vs SRR8334019""","""NextSeq 500 sequencing: GSM3518113: RNA-seq_K562_HS15_rep1 Homo sapiens RNA-Seq vs NextSeq 500 sequencing: GSM3518111: RNA-seq_K562_Ctrl_rep1 Homo sapiens RNA-Seq""","""The pause-initiation limit restricts transcription activation in human cells""","""SRA824487""","""cDNA""","""2019-07-11""","""6126887577 vs 6246893652""","""4528309758;4572681847 vs 4604709263;4642097213""","""ILLUMINA""","""Generic""","""RNA-seq_K562_HS15_rep1 vs RNA-seq_K562_Ctrl_rep1""","""SRP173667""","""100709060 vs 102213680""","""The pause-initiation limit restricts transcription activation in human cells""","""RNA-seq_K562_HS15_rep1

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 21. GSE115731 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date: 2025-05-04
- [Stress-Induced Low Complexity RNA Activates Physiological Amyloidogenesis](https://doi.org/10.1016/j.celrep.2018.07.040)
- コントロールと熱ショックのデータがしっかり揃っているデータなので収集する
- 

In [57]:
ID = "GSE115731"

GSE115731_PEP = pl.read_csv(
    f"../../Data/metadata/metadata_geofetch/{ID}/{ID}_PEP/{ID}_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("TruSeq Stranded Total RNA Library Prep Kit (without rRNA depletion)").alias("rna_library"), # add rna_library column from GEO accession (Illumina)
    pl.lit("https://doi.org/10.1016/j.celrep.2018.07.040").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("43 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("30 min.").alias("stress treatment time"), # add heat treatment time column
    pl.lit("37 C").alias("control treatment temperature"), # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

GSE115731_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE115731_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE115731_iseq_filtered = GSE115731_iseq[[s.name for s in GSE115731_iseq if not (s.null_count() == GSE115731_iseq.height)]]

GSE115731_combined = GSE115731_PEP.join(
    GSE115731_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE115731_combined = GSE115731_combined[[s.name for s in GSE115731_combined if not (s.null_count() == GSE115731_combined.height)]]

GSE115731_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE115731_modified_metadata.tsv",
    separator='\t'
)

# display(GSE115731_combined)

In [58]:
defined_pairs_gse115731 = [
    ("SRR7309387", "SRR7309386") # RNA from MCF7 cells, 30 minutes, 43C heat shock (30HS), no ribosomal RNA depletion vs RNA from MCF7 cells, no treatment (NT), no ribosomal RNA depletion
]

paired_metadata_df_gse115731 = create_paired_metadata(
    combined_df=GSE115731_combined,
    defined_sample_pairs=defined_pairs_gse115731,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse115731 is not None:
    paired_metadata_df_gse115731.write_csv(
        "../../Data/metadata/metadata_paired/GSE115731_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse115731)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_supplementary_file_2,gse_id,sample_data_row_count,experiment_accession,cell_line,isolate,disease,treatment,genome_build,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,isolate_right,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias,tissue_type
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR7309387""","""SRR7309386""","""rna_from_mcf7_cells_30_minutes_43c_heat_shock_30hs_no_ribosomal_rna_depletion vs rna_from_mcf7_cells_no_treatment_nt_no_ribosomal_rna_depletion""","""cDNA""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR7309387 vs SRR7309386""","""SRX4211755 vs SRX4211754""","""RNA from MCF7 cells, 30 minutes, 43C heat shock (30HS), no ribosomal RNA depletion vs RNA from MCF7 cells, no treatment (NT), no ribosomal RNA depletion""","""GSM3188431 vs GSM3188430""","""Public on Aug 02 2018""","""Jun 13 2018""","""Aug 02 2018""","""SRA""","""1""","""MCF7 mammary gland cell culture""","""Homo sapiens""","""9606""","""nuclear RNA""","""GPL18573""","""Stephen,,Lee""","""stephenlee@med.miami.edu""","""Gautier Bldg, Rm 228""","""Biochemistry & Molecular Biology""","""University of Miami""","""1011 NW 15 st.""","""Miami""","""Florida""","""33136""","""USA""","""Illumina NextSeq 500""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM3188431 vs GSM3188430""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3188nnn/GSM3188431/suppl/GSM3188431_06.GRCh38.starAligned.sortedByCoord.out.bed.gz vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3188nnn/GSM3188430/suppl/GSM31…","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3188nnn/GSM3188431/suppl/GSM3188431_06.GRCh38.starAligned.sortedByCoord.out_filtered.bed.gz vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3188nnn/GSM3188430/su…","""GSE115731""","""0""","""SRX4211755 vs SRX4211754""","""MCF7""","""HTB-22""","""adenocarcinoma""","""30 minutes, 43C heat shock vs none""","""GRCh38 (gencode v27)""","""BED, alignment files""","""SAMN09406962 vs SAMN09406963""","""TruSeq Stranded Total RNA Library Prep Kit (without rRNA depletion)""","""https://doi.org/10.1016/j.celrep.2018.07.040""","""43 C""","""30 min.""","""37 C""","""SRR7309387 vs SRR7309386""","""NextSeq 500 sequencing: GSM3188431: RNA from MCF7 cells, 30 minutes, 43C heat shock (30HS), no ribosomal RNA depletion Homo sapiens RNA-Seq vs NextSeq 500 sequencing: GSM3188430: RNA from MCF7 cells, …","""Heat shock-induced ribosomal intergenic spacer RNA""","""SRA721370""","""cDNA""","""2018-08-0

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE100469 (PAIRED) 4sU-seq

- validate date: 2025-05-04
- [HSV-1-induced disruption of transcription termination resembles a cellular stress response but selectively increases chromatin accessibility downstream of genes](https://doi.org/10.1371/journal.ppat.1006954)
- 4sU-seqなので、新生RNAを検出しているデータ
- 念の為収集
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に処理を移動

### GSE95854 (PAIRED) NO DOWNLOAD

- validate date: 2025-05-04
- [TimeLapse-seq: Adding a temporal dimension to RNA sequencing through nucleoside recoding](https://doi.org/10.1038/nmeth.4582)
- TimeLapse-seqという以前の研究で使用したデータ
- 今回も念の為収集しておく... と思ったが、ヒトのデータでは熱ストレスを加えているデータはなかったので今回は収集しない (マウスのデータには含まれていた)

In [59]:
# ID = "GSE95854"

# GSE95854_PEP = pl.read_csv(
#     f"../../Data/metadata/metadata_geofetch/{ID}/{ID}_PEP/{ID}_PEP_raw.csv",
#     separator=','
# ).filter(
#     pl.col("sample_organism_ch1") == "Homo sapiens"
# )

# pl.Config.set_fmt_str_lengths(200)
# display(GSE95854_PEP)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 22. GSE100408 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date: 2025-05-04
- [Probing the roles of SUMOylation in cancer cell biology by using a selective SAE inhibitor](https://doi.org/10.1038/nchembio.2463)
- 全文が見られないが、Gene Expression Omnibus (GEO) の ホームページから 条件を調べることが可能
- [GSM2680465](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2680465)
- 熱ストレスに関連するデータ (`0.5 h at 43 ⁰C`) とコントロールのデータのみ収集

In [60]:
ID = "GSE100408"

GSE100408_PEP = pl.read_csv(
    f"../../Data/metadata/metadata_geofetch/{ID}/{ID}_PEP/{ID}_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_title").str.contains(r'Heatshock')) | (pl.col("sample_title").str.contains(r'_No heatshock'))
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("TruSeq Stranded mRNA Library Kit").alias("rna_library"), # add rna_library column from GEO webpage (Illumina)
    pl.lit("https://doi.org/10.1038/nchembio.2463").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("43 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("30 min.").alias("stress treatment time"), # add heat treatment time column
    pl.lit("37 C (inferred)").alias("control treatment temperature"), # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

GSE100408_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE100408_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE100408_iseq_filtered = GSE100408_iseq[[s.name for s in GSE100408_iseq if not (s.null_count() == GSE100408_iseq.height)]]

GSE100408_combined = GSE100408_PEP.join(
    GSE100408_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE100408_combined = GSE100408_combined[[s.name for s in GSE100408_combined if not (s.null_count() == GSE100408_combined.height)]]

GSE100408_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE100408_modified_metadata.tsv",
    separator='\t'
)

# display(GSE100408_combined)

In [61]:
defined_pairs_gse100408 = [
    ("SRR5749642", "SRR5749648"), # HCT116_DMSO_With Heatshock (SAE032527) vs HCT116_DMSO_No heatshock (SAE034010)
    ("SRR5749643", "SRR5749649"), # HCT116_DMSO_With Heatshock (SAE032530) vs HCT116_DMSO_No heatshock (SAE034013)
    ("SRR5749644", "SRR5749650"), # HCT116_DMSO_With Heatshock (SAE032533) vs HCT116_DMSO_No heatshock (SAE034016)
    ("SRR5749645", "SRR5749651"), # HCT116_ML00754792_With Heatshock (SAE032537) vs HCT116_ML00754792_No heatshock (SAE034019)
    ("SRR5749646", "SRR5749652"), # HCT116_ML00754792_With Heatshock (SAE032540) vs HCT116_ML00754792_No heatshock (SAE034022)
    ("SRR5749647", "SRR5749653")  # HCT116_ML00754792_With Heatshock (SAE032542) vs HCT116_ML00754792_No heatshock (SAE034025)
]

paired_metadata_df_gse100408 = create_paired_metadata(
    combined_df=GSE100408_combined,
    defined_sample_pairs=defined_pairs_gse100408,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse100408 is not None:
    paired_metadata_df_gse100408.write_csv(
        "../../Data/metadata/metadata_paired/GSE100408_paired_metadata.tsv",
        separator='\t'
    )
    pl.Config.set_tbl_rows(2)
    display(paired_metadata_df_gse100408)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,treatment,dosage,time,genome_build,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR5749642""","""SRR5749648""","""hct116_dmso_with_heatshock_sae032527_ vs hct116_dmso_no_heatshock_sae034010_""","""cDNA""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR5749642 vs SRR5749648""","""SRX2949892 vs SRX2949898""","""HCT116_DMSO_With Heatshock (SAE032527) vs HCT116_DMSO_No heatshock (SAE034010)""","""GSM2680462 vs GSM2680468""","""Public on Sep 01 2017""","""Jun 23 2017""","""May 15 2019""","""SRA""","""1""","""HCT116_DMSO_With Heatshock vs HCT116_DMSO_No heatshock""","""Homo sapiens""","""9606""","""total RNA""","""SAE032527 vs SAE034010""","""GPL11154""","""Pooja,,Shah""","""pooja.shah@takeda.com""","""Takeda Pharmaceuticals""","""40 Landsdowne Street""","""Cambridge""","""MA""","""2139""","""USA""","""Illumina HiSeq 2000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM2680462 vs GSM2680468""","""NONE""","""GSE100408""","""0""","""SRX2949892 vs SRX2949898""","""colorectal cancer cell line HCT116""","""DMSO""","""null""","""3.5 hr""","""Human Genome (Human.B37.3)""","""mRNA FPKM count data for each sample""","""SAMN07270995 vs SAMN07270989""","""TruSeq Stranded mRNA Library Kit""","""https://doi.org/10.1038/nchembio.2463""","""43 C""","""30 min.""","""37 C (inferred)""","""SRR5749642 vs SRR5749648""","""Illumina HiSeq 2000 sequencing: GSM2680462: HCT116_DMSO_With Heatshock (SAE032527) Homo sapiens RNA-Seq vs Illumina HiSeq 2000 sequencing: GSM2680468: HCT116_DMSO_No heatshock (SAE034010) Homo sapiens…","""Homo sapiens""","""SRA580466""","""cDNA""","""2017-11-09""","""3624919442 vs 4791959715""","""2949088092;3055764235 vs 3736560023;3875821478""","""ILLUMINA""","""Generic""","""HCT116_DMSO_With Heatshock (SAE032527) vs HCT116_DMSO_No heatshock (SAE034010)""","""SRP110248""","""71480963 vs 83870328""","""Probing the Roles of SUMOylation in Cancer Cell Biology Using a Selective SAE inhibitor""","""HCT116_DMSO_With Heatshock (SAE032527) vs HCT116_DMSO_No heatshock (SAE034010)""","""Illumina HiSeq 2000""","""Illumina HiSeq 2000 sequencing: GSM2680462: HCT116_DMSO_With Heatshock (SAE032527) Homo sapiens RNA-Seq vs Illumina HiSeq 2000 sequencing: GSM2680468: HCT116_DMSO_No heatshock (SAE034010) Homo sapiens…","""7291058226 vs 8554773456""","""RNA-Seq""","""TRANSCRIP

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Concat dataframe (RNA-seq only)

In [62]:
df_rna_seq_true_no_control: list[pl.DataFrame | None] = [
    paired_metadata_df_gse228346_rna_seq,
    paired_metadata_df_gse255207,
    paired_metadata_df_gse270672,
    paired_metadata_df_gse197993,
    paired_metadata_df_gse137559,
    paired_metadata_df_gse130892_rna_seq,
    paired_metadata_df_gse132447_rna_seq,
    paired_metadata_df_gse289231,
    paired_metadata_df_gse279569,
    paired_metadata_df_gse285497,
    paired_metadata_df_gse279445,
    paired_metadata_df_gse229915,
    paired_metadata_df_gse242710,
    paired_metadata_df_gse221579,
    paired_metadata_df_gse164834,
    paired_metadata_df_gse182960,
    paired_metadata_df_gse198441,
    paired_metadata_df_gse164846,
    paired_metadata_df_gse149797,
    paired_metadata_df_gse123980_rna_seq,
    paired_metadata_df_gse115731,
    paired_metadata_df_gse100408
]

rna_seq_true_no_control: list[pl.DataFrame] = [
    df for df in df_rna_seq_true_no_control if df is not None
]

In [63]:
rna_seq_true_no_control = pl.concat(rna_seq_true_no_control, how="diagonal")

rna_seq_true_no_control.write_csv(
    f"../../Data/metadata/metadata_{VERSION}/rna_seq_true_no_control_metadata_{execute_date}.tsv",
    separator="\t"
)